# Carbon-, power- and network-aware scheduling: simulation notebook (free Colab)

**What this runs:** a two-level simulator. A cluster-level discrete-event simulator (queue, EASY backfill, GPU pods, policies **P0, P2, P3, P4**) consumes a **job profile table** (iteration time, compute fraction, inter-pod traffic). The profile table comes from ASTRA-sim when you have it, and from a clearly labelled synthetic placeholder until then. A fluid pod-uplink congestion model couples jobs that share the network; a phase-dependent power model plus carbon intensity turns it into energy and carbon.

**v7 (22 Oct 2026):** jobs spread over 3 or more pods use measured `spread_k3`/`spread_k4` profile rows when present, otherwise a flat-ring extension of the 2-pod row (`K_RULE`, consistent with the ns-3 ring all-reduce; v0.1.0's hierarchical rule overstated interleaved inter-pod traffic by k-1); new metric `gpuh_frac_k3plus`; new results tags `meas5_*`, so pre-v7 results are not reused.

**v6 (21 Oct 2026):** results, digest, figures and bundle are named by results tag, so the interleaved and aligned runs no longer overwrite each other; an input-fingerprint guard stops the run if a results file was written with different input files.

**How to use it (about 3 cells of effort):**
1. Set `MODE` in the first code cell: `"quick"` (about 2 to 4 minutes) or `"full"` (about 15 to 25 minutes on free Colab).
2. **Runtime > Run all.**
3. Copy the text block printed by the last cell (`=== RESULT DIGEST ===`) back to Claude. The zip it also builds contains everything else.

Everything checkpoints to Google Drive, so a disconnect costs you nothing: run all again and finished runs are skipped.

**Honesty box.** Until ASTRA-sim profiles, calibrated GPU power, and real carbon data are plugged in (optional hooks below), the numbers are *pipeline tests with ILLUSTRATIVE parameters*, not findings. The digest says so in its first lines.

In [ ]:
import os, sys, json, time, platform
MODE = "quick"          # "quick" or "full"
MASTER_SEED = 20261004  # recorded in the results; replications use SeedSequence streams derived from it

# ---- persistent storage (falls back to a local folder when not on Colab) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/msc_sim'
    ON_COLAB = True
except Exception:
    BASE = os.path.abspath('./msc_sim_local')
    ON_COLAB = False
for sub in ('results', 'data', 'figures', 'bundles'):
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print('storage:', BASE, '| colab:', ON_COLAB, '| python', platform.python_version(), '| cpus', os.cpu_count())

# ---- OPTIONAL real inputs: drop these files in BASE/data (all optional; absent = placeholder used) ----
PROFILE_VARIANT   = 'interleaved'   # 'interleaved' = WORST-CASE rank placement (every ring hop crosses pods); 'aligned' = BEST-CASE (one crossing). Report both: they bracket reality.
K_RULE = 'ring_interleaved' if PROFILE_VARIANT == 'interleaved' else 'ring_aligned'   # extends 2-pod rows to k>2 pods where no measured 'spread_k<k>' row exists (flat ring, as simulated in ns-3)
ASTRA_PROFILE_CSV = f'{BASE}/data/astra_profiles.csv' if PROFILE_VARIANT == 'interleaved' else f'{BASE}/data/astra_profiles_aligned.csv'   # jtype,n_gpus,pclass,iter_time_s,frac_compute,interpod_GB,source
GHANA_CI_CSV      = f'{BASE}/data/ghana_ci.csv'        # timestamp,gco2_per_kwh   (hourly)
TRACE_CSV         = f'{BASE}/data/trace.csv'           # arrival_s,n_gpus,duration_s  (production GPU trace)
TARGET_GPU_IDLE_W = 132    # W/GPU = (1860-800)/8: 8-GPU H100 node idle 1.86 kW (arXiv 2506.14551) minus ASSUMED 800 W host overhead (P_server)
TARGET_GPU_MAX_W  = 874    # W/GPU-slot = (7790-800)/8: highest observed node average 7.79 kW (same paper); includes the GPU's share of node fan/switch load
GPU_POWER_CSV     = f'{BASE}/data/gpu_power.csv'       # workload,frac_compute,power_w   (per-GPU measurements)

## 1. Simulator source (written next to the notebook)
Three small modules. Re-running this section overwrites them and reloads cleanly.

In [ ]:
%%writefile dcsim.py
"""
dcsim: cross-layer simulator for carbon-, power- and network-aware scheduling of AI training jobs.

Study type: TERMINATING discrete-event simulation (cluster starts empty, fixed accounting window).
Layers:
  1. cluster layer  : job arrivals, queue (FIFO + EASY backfill), GPU pods, policies P0/P2/P3/P4
  2. job layer      : per-job profile table (iter_time, compute fraction, inter-pod GB per iteration).
                      Source = ASTRA-sim output (preferred) or a synthetic analytic profile (placeholder).
  3. network layer  : fluid pod-uplink congestion with a calibratable slowdown curve (ns-3 hook)
  4. energy layer   : phase-dependent GPU power, idle GPUs, servers, network ports, PUE, carbon = energy x CI(t)

ALL numeric defaults marked ILLUSTRATIVE are placeholders, not measurements. They must be calibrated
and reported before any result is treated as a finding.
"""
from __future__ import annotations

import functools
import hashlib
import heapq
import json
import math
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd

__version__ = "0.2.0"
HOUR = 3600.0
J_PER_KWH = 3.6e6
MASTER_SEED_DEFAULT = 20261004

# ----------------------------------------------------------------------------------------------
# Parameters
# ----------------------------------------------------------------------------------------------


@dataclass
class PowerParams:
    """ILLUSTRATIVE placeholders. Calibrate P_gpu_* against measurements (reference [2])."""
    P_gpu_idle: float = 100.0         # W per GPU when idle or waiting on communication
    P_gpu_max: float = 650.0          # W per GPU in a pure compute phase
    P_server: float = 800.0           # W per server host overhead (always on)
    P_port: float = 10.0              # W per active uplink port / transceiver
    P_net_static_pod: float = 400.0   # W per pod switch (always on)
    P_core_static: float = 400.0      # W core fabric (OCS or spine, always on)
    PUE: float = 1.3                  # constant facility overhead (assumption, state and justify)


@dataclass
class ClusterParams:
    """ILLUSTRATIVE placeholders for a small training cluster (default 128 GPUs)."""
    n_pods: int = 4
    gpus_per_pod: int = 32
    gpus_per_server: int = 8
    fabric: str = "ocs"                 # 'ocs' | 'electrical' (see FABRICS)
    uplink_GBps: float = 50.0           # aggregate per-pod uplink capacity (GB/s)
    intra_GBps: float = 300.0           # effective intra-pod all-reduce bandwidth (GB/s)
    reconf_delay_s: float = 0.05        # OCS circuit set-up added when a job spans pods
    ports_per_pod: int = 8              # transceivers per pod uplink
    uplink_scale: float = 1.0           # multiplies the fabric preset's per-pod uplink (keeps uplink per GPU constant when pod size changes)
    sleep_idle_ports: bool = True       # uplink ports of pods with no inter-pod traffic power down
    # How a profile measured on 2 pods is extended to a job spread over k > 2 pods, when no measured row for that k exists.
    # 'hier'             hierarchical ring all-reduce, per-pod volume x 2(k-1)/k relative to k=2 (v0.1.0 behaviour; kept so
    #                    stored results reproduce). NOT consistent with the flat-ring ns-3 profiles.
    # 'ring_interleaved' flat ring, ranks round-robin over pods (every hop crosses): per-pod volume x 2/k
    # 'ring_aligned'     flat ring, ranks in contiguous blocks (one crossing hop per pod): per-pod volume x 1
    # Measured rows (pclass 'spread_k3', 'spread_k4', ...) always take precedence; the rule extends from the largest measured k.
    k_rule: str = "hier"
    # Congestion curve: (offered load / capacity) -> slowdown of the exposed communication time.
    # Default = fluid processor sharing. REPLACE with a curve fitted from ns-3 runs.
    congestion_pts: tuple = ((0.0, 1.0), (1.0, 1.0), (8.0, 8.0))
    power_cap_W: Optional[float] = None  # facility power cap (admission control), None = no cap
    tick_s: float = HOUR                 # scheduler re-evaluation period
    horizon_h: float = 168.0             # arrivals are generated in [0, horizon]
    window_h: float = 264.0              # accounting window (horizon + drain)

    @property
    def n_gpus(self) -> int:
        return self.n_pods * self.gpus_per_pod

    @property
    def n_servers(self) -> int:
        return self.n_gpus // self.gpus_per_server


# Fabric presets: the "optical" layer is an ABSTRACTION (capacity, port power, set-up delay),
# not a physical-layer or wavelength-level model.  ILLUSTRATIVE numbers.
FABRICS = {
    "ocs":        dict(uplink_GBps=50.0, reconf_delay_s=0.05, P_port=8.0,  P_core_static=300.0),
    "electrical": dict(uplink_GBps=50.0, reconf_delay_s=0.0,  P_port=15.0, P_core_static=900.0),
}


def apply_fabric(cluster: ClusterParams, power: PowerParams, name: str):
    f = FABRICS[name]
    cluster.fabric = name
    cluster.uplink_GBps = f["uplink_GBps"]
    cluster.reconf_delay_s = f["reconf_delay_s"]
    power.P_port = f["P_port"]
    power.P_core_static = f["P_core_static"]
    return cluster, power


# ----------------------------------------------------------------------------------------------
# Job types and profile table
# ----------------------------------------------------------------------------------------------

# ILLUSTRATIVE job mix. G_GB = gradient volume per iteration; t_comp = compute time per iteration.
JOB_TYPES = {
    "llm_large": dict(weight=0.15, gpus=(32, 64), gprob=(0.6, 0.4), G_GB=14.0, t_comp=2.0, overlap=0.5,
                      med_h=2.5, sigma=0.8),
    "llm_mid":   dict(weight=0.35, gpus=(8, 16, 32), gprob=(0.3, 0.4, 0.3), G_GB=4.0, t_comp=1.0, overlap=0.5,
                      med_h=1.5, sigma=0.9),
    "vision":    dict(weight=0.50, gpus=(8, 16), gprob=(0.6, 0.4), G_GB=0.4, t_comp=0.5, overlap=0.7,
                      med_h=1.0, sigma=0.8),
}


def kscale(k: int) -> float:
    """Per-pod uplink volume of a hierarchical ring all-reduce over k pods, relative to gradient size G."""
    return 2.0 * (k - 1) / k if k > 1 else 0.0


K_RULES = {
    "hier": lambda k: kscale(k) / kscale(2),
    "ring_interleaved": lambda k: 2.0 / k,
    "ring_aligned": lambda k: 1.0,
}


def spread_pclass(k: int) -> str:
    """Profile class name for a job spanning k pods: 'spread' for k=2 (as in every existing table), 'spread_k<k>' above."""
    return "spread" if k == 2 else f"spread_k{k}"


def synthetic_profile(cluster: ClusterParams, job_types=JOB_TYPES, comm_scale: float = 1.0) -> pd.DataFrame:
    """PLACEHOLDER profile from an analytic alpha-beta style model (hierarchical ring all-reduce with
    partial overlap). Replace with ASTRA-sim measurements: same schema, source='astra-sim'.
    interpod_GB = GB crossing EACH involved pod uplink per iteration, evaluated at k=2 pods."""
    rows = []
    for jt, p in job_types.items():
        G = p["G_GB"] * comm_scale
        for n in p["gpus"]:
            for pclass in ("compact", "spread"):
                if pclass == "compact" and n > cluster.gpus_per_pod:
                    continue
                k = 1 if pclass == "compact" else 2
                n_loc = n if k == 1 else n / 2.0
                comm_intra = 2.0 * (n_loc - 1) / n_loc * G / cluster.intra_GBps
                comm_inter = kscale(k) * G / cluster.uplink_GBps if k > 1 else 0.0
                comm = comm_intra + comm_inter
                hidden = min(comm, p["overlap"] * p["t_comp"])
                exposed = comm - hidden
                it = p["t_comp"] + exposed
                rows.append(dict(jtype=jt, n_gpus=n, pclass=pclass, iter_time_s=it,
                                 frac_compute=p["t_comp"] / it,
                                 interpod_GB=(kscale(2) * G if k > 1 else 0.0),
                                 comm_time_s=comm_inter, source="synthetic-parametric"))
    return pd.DataFrame(rows)


def profile_dict(df: pd.DataFrame) -> dict:
    """Profile table -> {(jtype, n_gpus, pclass): (iter_time_s, frac_compute, interpod_GB, comm_time_s)}.
    comm_time_s = time per iteration of the communication that crosses the (congestible) pod uplinks at no
    congestion, hidden or exposed. Optional column; if absent it defaults to the EXPOSED communication time
    (iter_time*(1-frac_compute)), which is conservative-low when communication is mostly hidden."""
    has = "comm_time_s" in df.columns
    out = {}
    for r in df.itertuples(index=False):
        it, fc = float(r.iter_time_s), float(r.frac_compute)
        comm = float(r.comm_time_s) if has and not pd.isna(r.comm_time_s) else it * (1.0 - fc)
        out[(r.jtype, int(r.n_gpus), r.pclass)] = (it, fc, float(r.interpod_GB), comm)
    return out


_ASTRA_LINE = None


def parse_astra_output(text: str):
    """Parse ASTRA-sim stdout lines 'sys[i] finished, N cycles, exposed communication M cycles'.
    Returns (cycles array, exposed array) ordered by NPU id."""
    import re
    rows = sorted((int(a), int(b), int(c)) for a, b, c in
                  re.findall(r"sys\[(\d+)\] finished, (\d+) cycles, exposed communication (\d+) cycles", text))
    return np.array([r[1] for r in rows], float), np.array([r[2] for r in rows], float)


def profile_row_from_astra(jtype, n_gpus, pclass, cycles_per_npu, exposed_per_npu, n_iterations,
                           interpod_GB, ns_per_cycle=1.0, comm_time_s=None):
    """Convert ASTRA-sim per-NPU output ('finished, N cycles, exposed communication M cycles') to a profile row.
    ns_per_cycle: ASSUMED 1.0 (analytical backend time unit = ns). VERIFY against the ASTRA-sim docs.
    n_iterations: number of training iterations contained in the simulated Chakra trace.
    comm_time_s: optional total (hidden+exposed) inter-pod communication time per iteration, e.g. from an
    ASTRA-sim run of the communication alone; if None the model uses the exposed time (conservative-low)."""
    total = float(np.max(cycles_per_npu))
    exposed = float(np.mean(exposed_per_npu))
    it_s = total * ns_per_cycle * 1e-9 / n_iterations
    fc = max(0.0, min(1.0, 1.0 - exposed / total))
    return dict(jtype=jtype, n_gpus=int(n_gpus), pclass=pclass, iter_time_s=it_s, frac_compute=fc,
                interpod_GB=float(interpod_GB), comm_time_s=(float(comm_time_s) if comm_time_s is not None else float("nan")),
                source="astra-sim")


# ----------------------------------------------------------------------------------------------
# Randomness, carbon intensity, workloads
# ----------------------------------------------------------------------------------------------

STREAMS = {"arrival": 1, "attrs": 2, "ci": 3, "forecast": 4, "calib": 5, "misc": 6}


def rng_for(master_seed: int, rep: int, stream: str) -> np.random.Generator:
    """Independent, well-separated streams via SeedSequence (never hand-picked seeds 1,2,3)."""
    return np.random.Generator(np.random.PCG64(np.random.SeedSequence([master_seed, rep, STREAMS[stream]])))


class CI:
    """Hourly step-function carbon intensity (gCO2/kWh) with exact interval integrals."""

    def __init__(self, arr):
        self.a = np.asarray(arr, float)
        self.n = len(self.a)
        self.cum = np.concatenate([[0.0], np.cumsum(self.a * HOUR)])

    def at(self, t: float) -> float:
        return float(self.a[min(int(t // HOUR), self.n - 1)])

    def _F(self, t):
        t = np.asarray(t, float)
        h = np.minimum((t // HOUR).astype(int), self.n - 1)
        return self.cum[h] + self.a[h] * (t - h * HOUR)

    def integral(self, t0, t1):
        """Integral of CI dt over [t0, t1] in (gCO2/kWh)*s. Extends the last value beyond the series."""
        return self._F(t1) - self._F(t0)

    def mean(self, t0, t1):
        return float(self.integral(t0, t1)) / max(t1 - t0, 1e-9)


def make_ci(scenario: str, rep: int, hours: int, master_seed: int = MASTER_SEED_DEFAULT) -> np.ndarray:
    """Carbon-intensity scenarios. The synthetic scenarios are STRUCTURAL stand-ins (low vs high variability),
    NOT any real grid. Use 'csv:<path>' (columns timestamp,gco2_per_kwh; hourly) for real data such as Ghana."""
    if scenario.startswith("csv:"):
        df = pd.read_csv(scenario[4:])
        col = [c for c in df.columns if c != df.columns[0]][0]
        v = df[col].astype(float).to_numpy()
        reps = int(np.ceil(hours / len(v)))
        return np.tile(v, reps)[:hours]
    rng = rng_for(master_seed, rep, "ci")
    h = np.arange(hours)
    hod = h % 24
    day = h // 24
    n_days = int(day.max()) + 1
    ar = np.zeros(n_days)
    for d in range(1, n_days):
        ar[d] = 0.6 * ar[d - 1] + rng.normal(0, 1)
    noise = np.zeros(hours)
    for i in range(1, hours):
        noise[i] = 0.9 * noise[i - 1] + rng.normal(0, 1)
    if scenario == "synthetic_low_var":
        base = 420.0
        v = base * (1 + 0.03 * np.cos(2 * np.pi * (hod - 19) / 24) + 0.01 * noise + 0.02 * ar[day])
    elif scenario == "synthetic_high_var":
        base = 330.0
        solar = np.exp(-0.5 * ((hod - 13) / 3.5) ** 2)
        evening = np.exp(-0.5 * ((hod - 19.5) / 2.0) ** 2)
        v = base * (1 - 0.65 * solar + 0.25 * evening + 0.04 * noise + 0.20 * ar[day] * (0.4 + solar))
    else:
        raise ValueError(f"unknown CI scenario {scenario}")
    return np.maximum(v, 20.0)


def ci_variability(ci) -> dict:
    ci = pd.Series(np.asarray(ci, float))
    hod = ci.groupby(np.arange(len(ci)) % 24).mean()
    day = np.arange(len(ci)) // 24
    rng_day = ci.groupby(day).max() - ci.groupby(day).min()
    return dict(mean=float(ci.mean()), cv=float(ci.std() / ci.mean()), min=float(ci.min()), max=float(ci.max()),
                max_over_min=float(ci.max() / ci.min()), mean_daily_range=float(rng_day.mean()),
                hour_of_day_swing=float(hod.max() - hod.min()),
                daily_range_over_mean=float(rng_day.mean() / ci.mean()))


@dataclass
class Job:
    jid: int
    arrival: float
    jtype: str
    n_gpus: int
    n_iters: float
    deferrable: bool
    deadline: float
    dur_ref_s: float
    # runtime state
    start: Optional[float] = None
    finish: Optional[float] = None
    remaining: float = 0.0
    pods: Optional[dict] = None
    pods_list: Optional[list] = None
    k: int = 1
    iter_unc: float = 0.0
    fc_unc: float = 1.0
    t_comp: float = 0.0
    gb_k: float = 0.0
    demand: float = 0.0
    s: float = 1.0
    iter_eff: float = 0.0
    finish_est: float = 0.0
    version: int = 0
    target: Optional[float] = None
    unc_runtime: float = 0.0
    comm_unc: float = 0.0
    ci_at_start: float = 0.0


def _ref_class(n, cluster):
    return "compact" if n <= cluster.gpus_per_pod else "spread"


def arrivals(model: str, lam: float, horizon_s: float, rng) -> np.ndarray:
    """Arrival times. Exponential gaps by inverse transform (-ln(1-U)/lam)."""
    out = []
    if model == "poisson":
        t = 0.0
        while True:
            t += -math.log1p(-rng.random()) / lam
            if t >= horizon_s:
                break
            out.append(t)
    elif model == "bursty":                 # 2-state MMPP, same long-run mean rate lam
        t_on, t_off, ratio = 2 * HOUR, 6 * HOUR, 0.1
        pi_on = t_on / (t_on + t_off)
        lam_on = lam / (pi_on + ratio * (1 - pi_on))
        lam_off = ratio * lam_on
        t, on = 0.0, rng.random() < pi_on
        while t < horizon_s:
            dur = -math.log1p(-rng.random()) * (t_on if on else t_off)
            end = min(t + dur, horizon_s)
            r = lam_on if on else lam_off
            u = t
            while True:
                u += -math.log1p(-rng.random()) / r
                if u >= end:
                    break
                out.append(u)
            t, on = end, not on
    elif model == "diurnal":                # NHPP by thinning, mean rate lam, peak at 14:00
        a = 0.8
        lmax = lam * (1 + a)
        t = 0.0
        while True:
            t += -math.log1p(-rng.random()) / lmax
            if t >= horizon_s:
                break
            lt = lam * (1 + a * math.cos(2 * math.pi * ((t / HOUR) % 24 - 14) / 24))
            if rng.random() < lt / lmax:
                out.append(t)
    else:
        raise ValueError(f"unknown arrival model {model}")
    return np.array(out)


def _draw_attrs(rng, job_types, cluster, prof, dur_lo=0.25, dur_hi=24.0):
    names = list(job_types)
    w = np.array([job_types[n]["weight"] for n in names])
    jt = names[int(rng.choice(len(names), p=w / w.sum()))]
    p = job_types[jt]
    n = int(p["gpus"][int(rng.choice(len(p["gpus"]), p=np.array(p["gprob"]) / np.sum(p["gprob"])))])
    dur_h = float(np.clip(math.exp(math.log(p["med_h"]) + p["sigma"] * rng.normal()), dur_lo, dur_hi))
    return jt, n, dur_h * HOUR



@functools.lru_cache(maxsize=16)
def expected_gpu_seconds(job_types_json: str, dur_lo=0.25, dur_hi=24.0, m=20001) -> float:
    """E[n_gpus x duration_s] per job, by deterministic quantile integration (no sampling noise)."""
    from scipy.stats import norm
    jt = json.loads(job_types_json)
    z = norm.ppf((np.arange(m) + 0.5) / m)
    wsum = sum(p["weight"] for p in jt.values())
    tot = 0.0
    for p in jt.values():
        dur = np.clip(np.exp(math.log(p["med_h"]) + p["sigma"] * z), dur_lo, dur_hi).mean() * HOUR
        en = sum(g * q for g, q in zip(p["gpus"], np.array(p["gprob"]) / np.sum(p["gprob"])))
        tot += p["weight"] / wsum * en * dur
    return float(tot)


def make_workload(rep: int, arrival_model: str, util: float, cluster: ClusterParams, profile: dict,
                  job_types=JOB_TYPES, master_seed: int = MASTER_SEED_DEFAULT, deferrable_frac: float = 0.7,
                  slack_range=(2.0, 12.0)):
    """Common-random-numbers workload: depends only on (master_seed, rep, arrival_model, util, params),
    never on the policy. Nominal utilisation = offered GPU-seconds / capacity at reference profile."""
    e_gpu_s = expected_gpu_seconds(json.dumps(job_types, sort_keys=True))
    lam = util * cluster.n_gpus / e_gpu_s
    horizon_s = cluster.horizon_h * HOUR
    rng_a = rng_for(master_seed, rep, "arrival")
    rng_j = rng_for(master_seed, rep, "attrs")
    times = arrivals(arrival_model, lam, horizon_s, rng_a)
    jobs = []
    for i, t in enumerate(times):
        jt, n, dur = _draw_attrs(rng_j, job_types, cluster, profile)
        deferrable = bool(rng_j.random() < deferrable_frac)
        slack = slack_range[0] + (slack_range[1] - slack_range[0]) * rng_j.random()
        it, fc, gb = profile[(jt, n, _ref_class(n, cluster))][:3]
        n_iters = dur / it
        deadline = t + dur * (1.0 + slack) if deferrable else t + dur * 1.5
        jobs.append(Job(jid=i, arrival=float(t), jtype=jt, n_gpus=n, n_iters=n_iters, deferrable=deferrable,
                        deadline=float(deadline), dur_ref_s=dur))
    return jobs, dict(lam_per_h=lam * HOUR, e_gpu_hours=e_gpu_s / HOUR)



def workload_from_trace(df, rep, util, cluster, profile, job_types=JOB_TYPES, master_seed=MASTER_SEED_DEFAULT,
                        cols=None, deferrable_frac=0.7, slack_range=(2.0, 12.0)):
    """Trace-driven workload (e.g. a production GPU-cluster trace). Replication `rep` = a different window of the trace.
    Consecutive jobs are taken from the window start until the offered GPU-seconds equal util x capacity x horizon, then
    arrival times are rescaled linearly to span the horizon (keeps the trace's burstiness shape).
    Traces carry no communication information: job type (hence communication intensity) is ASSIGNED at random per
    GPU count using the job-type weights. State this limitation wherever results are reported."""
    cols = cols or dict(arrival="arrival_s", n_gpus="n_gpus", duration="duration_s")
    d = df[[cols["arrival"], cols["n_gpus"], cols["duration"]]].dropna().sort_values(cols["arrival"]).reset_index(drop=True)
    d.columns = ["arrival", "n_gpus", "duration"]
    sizes = sorted({g for p in job_types.values() for g in p["gpus"]})
    d["n"] = d["n_gpus"].apply(lambda x: min(sizes, key=lambda s: abs(s - x)))
    d["dur"] = d["duration"].clip(0.25 * HOUR, 24 * HOUR)
    target = util * cluster.n_gpus * cluster.horizon_h * HOUR
    start = (rep * 7919) % max(len(d) - 1, 1)
    idx = np.concatenate([np.arange(start, len(d)), np.arange(0, start)])      # wrap around
    d["arr_u"] = d["arrival"]
    if start > 0:                       # unwrap: the wrapped head of the trace continues AFTER its tail
        gap = float(np.median(np.diff(d["arrival"].to_numpy()))) if len(d) > 2 else 1.0
        d.loc[:start - 1, "arr_u"] = d.loc[:start - 1, "arrival"] + (d["arrival"].iloc[-1] - d["arrival"].iloc[0]) + gap
    cum = np.cumsum((d.loc[idx, "n"] * d.loc[idx, "dur"]).to_numpy())
    m = int(np.searchsorted(cum, target)) + 1
    sel = d.loc[idx[:min(m, len(idx))]].copy()
    t0, t1 = sel["arr_u"].iloc[0], sel["arr_u"].iloc[-1]
    span = max(t1 - t0, 1.0)
    sel["t"] = (sel["arr_u"] - t0) / span * (cluster.horizon_h * HOUR * 0.999)
    rng = rng_for(master_seed, rep, "attrs")
    jobs = []
    for i, r in enumerate(sel.itertuples(index=False)):
        cands = [jt for jt, p in job_types.items() if r.n in p["gpus"]]
        w = np.array([job_types[c]["weight"] for c in cands])
        jt = cands[int(rng.choice(len(cands), p=w / w.sum()))]
        deferrable = bool(rng.random() < deferrable_frac)
        slack = slack_range[0] + (slack_range[1] - slack_range[0]) * rng.random()
        it, fc, gb = profile[(jt, int(r.n), _ref_class(int(r.n), cluster))][:3]
        dur = float(r.dur)
        jobs.append(Job(jid=i, arrival=float(r.t), jtype=jt, n_gpus=int(r.n), n_iters=dur / it, deferrable=deferrable,
                        deadline=float(r.t + dur * (1.0 + slack)) if deferrable else float(r.t + dur * 1.5),
                        dur_ref_s=dur))
    return jobs, dict(lam_per_h=len(jobs) / cluster.horizon_h, e_gpu_hours=float((sel["n"] * sel["dur"]).mean() / HOUR))


def workload_hash(jobs) -> str:
    h = hashlib.sha256()
    for j in jobs:
        h.update(f"{j.jid}|{j.arrival:.6f}|{j.jtype}|{j.n_gpus}|{j.n_iters:.6f}|{j.deferrable}|{j.deadline:.3f}".encode())
    return h.hexdigest()[:16]


# ----------------------------------------------------------------------------------------------
# Simulator
# ----------------------------------------------------------------------------------------------


class Sim:
    def __init__(self, jobs, profile: dict, cluster: ClusterParams, power: PowerParams, ci_true, policy,
                 ci_forecast=None, record_trace=False, check=False):
        self.cp, self.pp = cluster, power
        self.profile = profile
        self.jobs = jobs
        self.jobs_by_id = {j.jid: j for j in jobs}
        self.ci = CI(ci_true)
        self.ci_f = CI(ci_forecast if ci_forecast is not None else ci_true)
        self.policy = policy
        self.record_trace = record_trace
        self.check = check
        self.W = cluster.window_h * HOUR
        P = cluster.n_pods
        self.free = [cluster.gpus_per_pod] * P
        self.queue: list[Job] = []
        self.running: dict[int, Job] = {}
        self.done: list[Job] = []
        self.heap: list = []
        self.seq = 0
        self.t = 0.0
        self.last_t = 0.0
        self.load = np.zeros(P)
        self.E_J = 0.0            # total energy (J, after PUE)
        self.carbon_g = 0.0
        self.area_n = 0.0
        self.area_busy = 0.0
        self.peak_W = 0.0
        self.cap_exceed_s = 0.0
        self.cap_block_events = 0
        self.trace = []
        self.parts_J = dict(gpu_alloc=0.0, gpu_idle=0.0, server=0.0, net=0.0)   # pre-PUE
        self.max_spread_running = 0
        self.full_reeval = False
        self._memo = {}
        xs, ys = zip(*cluster.congestion_pts)
        self._cx, self._cy = np.array(xs, float), np.array(ys, float)
        self.P = None
        self._power_cache()
        for j in jobs:
            self._push(j.arrival, "arr", j.jid)
        self._push(0.0, "tick", None)

    # ---- events ----
    def _push(self, t, typ, payload):
        self.seq += 1
        heapq.heappush(self.heap, (t, self.seq, typ, payload))

    # ---- network model ----
    def curve(self, ratio):
        r = np.asarray(ratio, float)
        y = np.interp(r, self._cx, self._cy)
        last_x, last_y = self._cx[-1], self._cy[-1]
        return np.where(r > last_x, last_y * r / last_x, y)

    def slowdowns(self, specs):
        """specs: list of (key, pods_list, demand_GBps). Returns (dict key->s, load array)."""
        key = tuple((k, tuple(p), round(d, 9)) for k, p, d in specs)
        hit = self._memo.get(key)
        if hit is not None:
            return hit
        load = np.zeros(self.cp.n_pods)
        for k, pods, d in specs:
            if len(pods) > 1:
                for p in pods:
                    load[p] += d
        sp = self.curve(load / self.cp.uplink_GBps)
        out = {}
        for k, pods, d in specs:
            out[k] = float(max(sp[p] for p in pods)) if len(pods) > 1 else 1.0
        if len(self._memo) > 200000:
            self._memo.clear()
        self._memo[key] = (out, load)
        return out, load

    # ---- profile helpers ----
    def prof(self, job, k):
        """(iter_time, frac_compute, GB per pod uplink per iteration, congestible comm time) for k pods.
        Uses the measured row for this k if the table has one; otherwise the row for the largest measured k0 < k,
        with volume and communication time scaled by K_RULES[k_rule](k) / K_RULES[k_rule](k0)."""
        if k == 1:
            v = self.profile[(job.jtype, job.n_gpus, "compact")]
            return v[0], v[1], 0.0, 0.0
        k0 = k
        while k0 > 2 and (job.jtype, job.n_gpus, spread_pclass(k0)) not in self.profile:
            k0 -= 1
        v = self.profile[(job.jtype, job.n_gpus, spread_pclass(k0))]
        it, fc, gb = v[0], v[1], v[2]
        comm = v[3] if len(v) > 3 else it * (1.0 - fc)
        if k0 == k:
            return it, fc, gb, comm
        rule = K_RULES[self.cp.k_rule]
        f = rule(k) / rule(k0)
        if self.cp.k_rule == "hier":                # v0.1.0 behaviour, unchanged: iteration time of the 2-pod row
            return it, fc, gb * f, comm * f
        # flat-ring rules: the ns-3 profiles have no overlap (it = t_comp + comm), so the uncongested iteration time
        # moves with the communication time; communication never falls below the compact (single-pod) value
        vc = self.profile[(job.jtype, job.n_gpus, "compact")]
        comm_c = vc[3] if len(vc) > 3 else vc[0] * (1.0 - vc[1])
        comm_k = max(comm * f, comm_c)
        t_comp = it * fc
        it_k = t_comp + (it - t_comp) * (comm_k / comm if comm > 0 else 1.0)
        return it_k, t_comp / it_k, gb * f, comm_k

    @staticmethod
    def it_eff(it, comm, s):
        """Congested iteration time: only the congestible communication stretches (conservative: no overlap slack)."""
        return it + comm * (s - 1.0)

    def gpu_power(self, n, fc):
        return n * (self.pp.P_gpu_idle + (self.pp.P_gpu_max - self.pp.P_gpu_idle) * fc)

    # ---- power ----
    def _power_cache(self):
        pp, cp = self.pp, self.cp
        alloc = 0
        gpu_alloc = 0.0
        for j in self.running.values():
            alloc += j.n_gpus
            gpu_alloc += self.gpu_power(j.n_gpus, j.t_comp / j.iter_eff)
        gpu_idle = (cp.n_gpus - alloc) * pp.P_gpu_idle
        server = cp.n_servers * pp.P_server
        n_active = int(np.sum(self.load > 1e-12)) if cp.sleep_idle_ports else cp.n_pods
        net = n_active * cp.ports_per_pod * pp.P_port + cp.n_pods * pp.P_net_static_pod + pp.P_core_static
        self.P = dict(gpu_alloc=gpu_alloc, gpu_idle=gpu_idle, server=server, net=net, alloc=alloc,
                      total=pp.PUE * (gpu_alloc + gpu_idle + server + net), n_active=n_active)

    def idle_floor_W(self):
        pp, cp = self.pp, self.cp
        return pp.PUE * (cp.n_gpus * pp.P_gpu_idle + cp.n_servers * pp.P_server
                         + cp.n_pods * pp.P_net_static_pod + pp.P_core_static
                         + (0 if cp.sleep_idle_ports else cp.n_pods * cp.ports_per_pod * pp.P_port))

    def max_power_W(self):
        pp, cp = self.pp, self.cp
        return pp.PUE * (cp.n_gpus * pp.P_gpu_max + cp.n_servers * pp.P_server
                         + cp.n_pods * (pp.P_net_static_pod + cp.ports_per_pod * pp.P_port) + pp.P_core_static)

    def est_total_power(self, active_jobs, extra_dyn_W=0.0, extra_pods=()):
        """Cluster power if only `active_jobs` run (+ extra dynamic W). For cap checks and forecasts."""
        pp, cp = self.pp, self.cp
        specs = [(j.jid, j.pods_list, j.demand) for j in active_jobs]
        s, load = self.slowdowns(specs) if specs else ({}, np.zeros(cp.n_pods))
        gpu_dyn = 0.0
        for j in active_jobs:
            ie = self.it_eff(j.iter_unc, j.comm_unc, s[j.jid])
            gpu_dyn += j.n_gpus * (pp.P_gpu_max - pp.P_gpu_idle) * (j.t_comp / ie)
        act = set(np.nonzero(load > 1e-12)[0].tolist()) | set(extra_pods)
        n_active = len(act) if cp.sleep_idle_ports else cp.n_pods
        net = n_active * cp.ports_per_pod * pp.P_port + cp.n_pods * pp.P_net_static_pod + pp.P_core_static
        base = cp.n_gpus * pp.P_gpu_idle + cp.n_servers * pp.P_server
        return pp.PUE * (base + gpu_dyn + net) + extra_dyn_W

    def cap_now(self):
        return self.cp.power_cap_W

    # ---- time advance ----
    def _advance(self, t):
        dt = t - self.last_t
        if dt > 0:
            for j in self.running.values():
                j.remaining -= dt / j.iter_eff
            P = self.P
            self.E_J += P["total"] * dt
            self.carbon_g += P["total"] * float(self.ci.integral(self.last_t, t)) / J_PER_KWH   # exact over CI steps inside (last_t, t]
            self.area_n += (len(self.queue) + len(self.running)) * dt
            self.area_busy += P["alloc"] * dt
            pue = self.pp.PUE
            self.parts_J["gpu_alloc"] += pue * P["gpu_alloc"] * dt
            self.parts_J["gpu_idle"] += pue * P["gpu_idle"] * dt
            self.parts_J["server"] += pue * P["server"] * dt
            self.parts_J["net"] += pue * P["net"] * dt
            self.peak_W = max(self.peak_W, P["total"])
            cap = self.cap_now()
            if cap is not None and P["total"] > cap + 1e-6:
                self.cap_exceed_s += dt
            if self.record_trace:
                self.trace.append((self.last_t, t, P["total"], P["gpu_alloc"], P["gpu_idle"], P["server"], P["net"]))
        self.last_t = t
        self.t = t

    def _refresh(self):
        """Recompute congestion slowdowns, completion events and power after any state change."""
        specs = [(j.jid, j.pods_list, j.demand) for j in self.running.values()]
        s, load = self.slowdowns(specs) if specs else ({}, np.zeros(self.cp.n_pods))
        self.load = load
        n_spread = 0
        for j in self.running.values():
            j.s = s[j.jid]
            j.iter_eff = self.it_eff(j.iter_unc, j.comm_unc, j.s)
            j.finish_est = self.t + max(j.remaining, 0.0) * j.iter_eff
            j.version += 1
            self._push(j.finish_est, "fin", (j.jid, j.version))
            n_spread += 1 if j.k > 1 else 0
        self.max_spread_running = max(self.max_spread_running, n_spread)
        self._power_cache()

    # ---- placement ----
    def first_fit(self, n):
        if sum(self.free) < n:
            return None
        rem, pl = n, {}
        for p in range(self.cp.n_pods):
            take = min(self.free[p], rem)
            if take > 0:
                pl[p] = take
                rem -= take
            if rem == 0:
                break
        return pl

    def candidates(self, job, free=None, load=None):
        free = self.free if free is None else free
        n = job.n_gpus
        if sum(free) < n:
            return []
        P = self.cp.n_pods
        comp = [p for p in range(P) if free[p] >= n]
        if comp:
            p = min(comp, key=lambda q: (free[q], q))
            return [{p: n}]
        load = self.load if load is None else load
        orders = [sorted(range(P), key=lambda q: (-free[q], load[q], q)),
                  sorted([q for q in range(P) if free[q] > 0], key=lambda q: (load[q], -free[q], q))]
        out, seen = [], set()
        for order in orders:
            rem, pl = n, {}
            for q in order:
                if free[q] <= 0:
                    continue
                take = min(free[q], rem)
                pl[q] = take
                rem -= take
                if rem == 0:
                    break
            key = tuple(sorted(pl.items()))
            if rem == 0 and key not in seen:
                seen.add(key)
                out.append(pl)
        return out

    def forecast_free(self, t):
        free = list(self.free)
        for j in self.running.values():
            if j.finish_est <= t:
                for p, c in j.pods.items():
                    free[p] += c
        return free

    def active_at(self, t):
        return [j for j in self.running.values() if j.finish_est > t + 1e-9]

    # ---- evaluation of a candidate (used by P3/P4 and the cap check) ----
    def eval_candidate(self, job, placement, t=None, active=None):
        """Estimated runtime, job dynamic power and externality (extra seconds imposed on running jobs)."""
        t = self.t if t is None else t
        active = list(self.running.values()) if active is None else active
        pods = sorted(placement)
        k = len(pods)
        it, fc, gb, comm = self.prof(job, k)
        specs = [(j.jid, j.pods_list, j.demand) for j in active]
        s_wo, _ = self.slowdowns(specs) if specs else ({}, None)
        cand = ("cand", pods, gb / it if k > 1 else 0.0)
        s_w, _ = self.slowdowns(specs + [cand])
        sc = s_w["cand"]
        ie = self.it_eff(it, comm, sc)
        R = job.n_iters * ie + (self.cp.reconf_delay_s if k > 1 else 0.0)
        t_comp = it * fc
        p_gpu = self.gpu_power(job.n_gpus, t_comp / ie)
        extra = []
        for j in active:
            if abs(s_w[j.jid] - s_wo[j.jid]) < 1e-12:
                continue
            ie_w = self.it_eff(j.iter_unc, j.comm_unc, s_w[j.jid])
            ie_wo = self.it_eff(j.iter_unc, j.comm_unc, s_wo[j.jid])
            rem_it = j.remaining if t <= self.t + 1e-9 else max(0.0, (j.finish_est - t) / j.iter_eff)
            extra.append((j, rem_it * (ie_w - ie_wo)))
        return dict(R=R, p_gpu=p_gpu, ie=ie, s=sc, k=k, extra=extra, pods=pods, gb=gb, it=it)

    def power_ok(self, job, placement):
        cap = self.cap_now()
        if cap is None:
            return True
        ev = self.eval_candidate(job, placement)
        net_extra = 0.0
        if self.cp.sleep_idle_ports and ev["k"] > 1:
            newp = [p for p in ev["pods"] if self.load[p] <= 1e-12]
            net_extra = len(newp) * self.cp.ports_per_pod * self.pp.P_port
        dW = self.pp.PUE * (ev["p_gpu"] - job.n_gpus * self.pp.P_gpu_idle + net_extra)
        ok = self.P["total"] + dW <= cap + 1e-6
        if not ok:
            self.cap_block_events += 1
        return ok

    # ---- lifecycle ----
    def _start(self, job, placement):
        pods = sorted(placement)
        k = len(pods)
        it, fc, gb, comm = self.prof(job, k)
        job.pods, job.pods_list, job.k = dict(placement), pods, k
        job.iter_unc, job.fc_unc, job.t_comp, job.comm_unc = it, fc, it * fc, comm
        job.gb_k = gb
        job.demand = gb / it if k > 1 else 0.0
        job.remaining = job.n_iters + (self.cp.reconf_delay_s / it if k > 1 else 0.0)
        job.unc_runtime = job.remaining * it
        job.start = self.t
        job.ci_at_start = self.ci.at(self.t)
        for p, c in placement.items():
            self.free[p] -= c
        self.queue.remove(job)
        self.running[job.jid] = job

    def _finish(self, job):
        job.finish = self.t
        job.remaining = 0.0
        for p, c in job.pods.items():
            self.free[p] += c
        del self.running[job.jid]
        self.done.append(job)

    def _reservation(self, job, power=False):
        fe = sorted((j.finish_est, j.n_gpus) for j in self.running.values())
        free = sum(self.free)
        if power and free >= job.n_gpus:
            return (fe[0][0] if fe else math.inf), 0
        for t_f, n in fe:
            free += n
            if free >= job.n_gpus:
                return t_f, free - job.n_gpus
        return math.inf, 0

    def _schedule(self):
        shadow, extra = None, 0
        for job in list(self.queue):
            status, placement = self.policy.decide(self, job)
            if status == "defer":
                continue
            if status == "blocked" or placement is None:
                if shadow is None:
                    shadow, extra = self._reservation(job)
                continue
            if shadow is not None:
                it = self.prof(job, len(placement))[0]
                est_fin = self.t + job.n_iters * it
                if est_fin <= shadow + 1e-9:
                    pass
                elif job.n_gpus <= extra:
                    extra -= job.n_gpus
                else:
                    continue
            if not self.power_ok(job, placement):
                if shadow is None:
                    shadow, extra = self._reservation(job, power=True)
                continue
            self._start(job, placement)
            self._refresh()

    def _check_invariants(self):
        alloc = [0] * self.cp.n_pods
        for j in self.running.values():
            for p, c in j.pods.items():
                alloc[p] += c
        for p in range(self.cp.n_pods):
            assert self.free[p] >= 0, "negative free GPUs"
            assert self.free[p] + alloc[p] == self.cp.gpus_per_pod, "GPU conservation violated"
        ids = [j.jid for j in self.queue] + list(self.running) + [j.jid for j in self.done]
        assert len(ids) == len(set(ids)), "job in two states"

    def run(self):
        arrived = 0
        while self.heap:
            t, _, typ, pl = heapq.heappop(self.heap)
            if t > self.W:
                break
            if typ == "fin":
                jid, ver = pl
                j = self.running.get(jid)
                if j is None or j.version != ver:
                    continue
            self._advance(t)
            self.full_reeval = False
            if typ == "arr":
                self.queue.append(self.jobs_by_id[pl])
                arrived += 1
            elif typ == "fin":
                self._finish(self.running[pl[0]])
                self._refresh()
            elif typ == "tick":
                self.full_reeval = True
                if t + self.cp.tick_s <= self.W:
                    self._push(t + self.cp.tick_s, "tick", None)
            self._schedule()
            if self.check:
                self._check_invariants()
        self._advance(self.W)
        return self.metrics()

    # ---- metrics ----
    def metrics(self):
        cp, pp = self.cp, self.pp
        done = self.done
        n_all = len(self.jobs)
        floor_W = self.idle_floor_W()
        floor_carbon = floor_W * float(self.ci.integral(0.0, self.W)) / J_PER_KWH
        floor_energy_kWh = floor_W * self.W / J_PER_KWH
        out = dict(n_jobs=n_all, n_done=len(done), n_incomplete=n_all - len(done))
        out["completion_rate"] = len(done) / max(n_all, 1)
        out["done_gpu_h"] = float(sum(j.n_gpus * j.dur_ref_s for j in done) / HOUR)
        out["energy_kWh"] = self.E_J / J_PER_KWH
        out["carbon_kg"] = self.carbon_g / 1000.0
        out["energy_dyn_kWh"] = out["energy_kWh"] - floor_energy_kWh
        out["carbon_dyn_kg"] = (self.carbon_g - floor_carbon) / 1000.0
        out["carbon_per_gpuh_g"] = self.carbon_g / max(out["done_gpu_h"], 1e-9)      # total carbon per COMPLETED GPU-hour
        out["peak_kW"] = self.peak_W / 1000.0
        out["mean_kW"] = self.E_J / self.W / 1000.0
        out["cap_exceed_h"] = self.cap_exceed_s / HOUR
        out["cap_block_events"] = self.cap_block_events
        out["util_time_avg"] = self.area_busy / (self.W * cp.n_gpus)
        if done:
            jct = np.array([j.finish - j.arrival for j in done])
            wait = np.array([j.start - j.arrival for j in done])
            run = np.array([j.finish - j.start for j in done])
            unc = np.array([j.unc_runtime for j in done])
            gp = np.array([j.n_gpus for j in done], float)
            out["jct_mean_h"] = float(jct.mean() / HOUR)
            out["jct_p95_h"] = float(np.percentile(jct, 95) / HOUR)
            out["wait_mean_h"] = float(wait.mean() / HOUR)
            out["net_inflation"] = float((run * gp).sum() / (unc * gp).sum())
            out["spread_frac"] = float(np.mean([j.k > 1 for j in done]))
            out["gpuh_frac_k3plus"] = float(np.sum([j.n_gpus * j.dur_ref_s for j in done if j.k >= 3]) /
                                            np.sum([j.n_gpus * j.dur_ref_s for j in done]))      # exposure to the k>2 extension
            out["k_max"] = int(max(j.k for j in done))
            out["ci_at_start_gpuw"] = float(np.sum([j.ci_at_start * j.n_gpus * j.dur_ref_s for j in done]) /
                                            np.sum([j.n_gpus * j.dur_ref_s for j in done]))
        viol = sum(1 for j in done if j.finish > j.deadline) + (n_all - len(done))
        out["deadline_viol_frac"] = viol / max(n_all, 1)
        out["max_spread_running"] = self.max_spread_running
        out["area_n"] = self.area_n
        return out


# ----------------------------------------------------------------------------------------------
# Policies. decide() -> ('start', placement) | ('defer', None) | ('blocked', None)
# ----------------------------------------------------------------------------------------------


class Policy:
    name = "base"

    def decide(self, sim, job):
        raise NotImplementedError


class P0(Policy):
    """Baseline: FIFO + EASY backfill, first-fit GPU fill, no carbon/network/power awareness (cap is a hard limit)."""
    name = "P0"

    def decide(self, sim, job):
        pl = sim.first_fit(job.n_gpus)
        return ("start", pl) if pl else ("blocked", None)


def _oblivious_est(sim, job):
    """Network-oblivious estimate: reference class profile, no congestion."""
    cls = _ref_class(job.n_gpus, sim.cp)
    it, fc, gb = sim.profile[(job.jtype, job.n_gpus, cls)][:3]
    R = job.n_iters * it
    dynW = sim.pp.PUE * job.n_gpus * (sim.pp.P_gpu_max - sim.pp.P_gpu_idle) * fc
    return R, dynW


def _start_grid(sim, job, R, margin=0.15):
    now = sim.t
    latest = job.deadline - R * (1.0 + margin)
    last_ok = sim.W - R * 1.2
    latest = min(latest, last_ok)
    if (not job.deferrable) or latest <= now + 1e-9:
        return np.array([now])
    t0 = math.floor(now / HOUR + 1e-9) * HOUR + HOUR
    grid = np.arange(t0, latest + 1e-9, HOUR)
    return np.concatenate([[now], grid[:96]])


class P2(Policy):
    """Carbon-aware: shift deferrable jobs to the lowest-forecast-carbon start within deadline slack, but only if the
    forecast saving is at least `min_gain` (default 5%); this avoids a straw-man baseline that delays jobs for
    negligible gains. 'P2raw' uses min_gain=0 (pure carbon minimiser).
    Network- and power-oblivious placement (first-fit)."""
    name = "P2"

    def __init__(self, min_gain: float = 0.05):
        self.min_gain = min_gain

    def decide(self, sim, job):
        if job.deferrable and job.target is not None and not sim.full_reeval \
                and sim.t < job.target - sim.cp.tick_s - 1e-9:
            return ("defer", None)
        R, dynW = _oblivious_est(sim, job)
        ts = _start_grid(sim, job, R)
        if len(ts) > 1:
            carb = dynW / 1000.0 * (sim.ci_f.integral(ts, ts + R)) / HOUR
            i = int(np.argmin(carb))               # ties -> earliest
            best = ts[i] if carb[i] < carb[0] * (1.0 - self.min_gain) - 1e-12 else ts[0]
        else:
            best = ts[0]
        job.target = float(best)
        if best > sim.t + 1e-9:
            return ("defer", None)
        pl = sim.first_fit(job.n_gpus)
        return ("start", pl) if pl else ("blocked", None)


class P3(Policy):
    """Network-aware: start ASAP, choose placement minimising own runtime plus interference imposed on others."""
    name = "P3"

    def decide(self, sim, job):
        cands = sim.candidates(job)
        if not cands:
            return ("blocked", None)
        best, bc = None, math.inf
        for pl in cands:
            ev = sim.eval_candidate(job, pl)
            c = job.n_gpus * ev["R"] + sum(j.n_gpus * x for j, x in ev["extra"])
            if c < bc - 1e-9:
                best, bc = pl, c
        return ("start", best)


class P4(Policy):
    """Joint carbon + network + power. Cost in gCO2-equivalent:
         carbon(start, runtime incl. congestion) + externality on running jobs + w_delay x delay price.
    Considers start-now (best placement) versus hourly future starts (placement class from forecast free GPUs,
    congestion from the jobs still active then). Power cap enforced at start and in future-start feasibility."""
    name = "P4"

    def __init__(self, w_delay: float = 0.02, defer_margin: float = 0.005, claim_aware: bool = True):
        self.w_delay = w_delay
        self.defer_margin = defer_margin
        self.claim_aware = claim_aware      # False = ablation 'P4nc': ignore GPUs already claimed by earlier queued jobs

    def _cost(self, sim, job, t, pl, active, ci_ref):
        pp = sim.pp
        ev = sim.eval_candidate(job, pl, t=t, active=active)
        net_dW = 0.0
        if sim.cp.sleep_idle_ports and ev["k"] > 1:
            act_load = sim.slowdowns([(j.jid, j.pods_list, j.demand) for j in active])[1] if active else np.zeros(sim.cp.n_pods)
            net_dW = sum(1 for p in ev["pods"] if act_load[p] <= 1e-12) * sim.cp.ports_per_pod * pp.P_port
        dynW = pp.PUE * (ev["p_gpu"] - job.n_gpus * pp.P_gpu_idle + net_dW)
        carbon = dynW / 1000.0 * float(sim.ci_f.integral(t, t + ev["R"])) / HOUR
        price_kW = pp.PUE * pp.P_gpu_max / 1000.0
        delay_h = (t + ev["R"] - job.arrival) / HOUR
        delay_cost = self.w_delay * job.n_gpus * price_kW * ci_ref * delay_h
        ext = 0.0
        for j, x in ev["extra"]:
            kW_j = pp.PUE * j.n_gpus * (pp.P_gpu_max - pp.P_gpu_idle) * (j.t_comp / j.iter_eff) / 1000.0
            ext += x / HOUR * (kW_j * ci_ref + self.w_delay * j.n_gpus * price_kW * ci_ref)
        return carbon + delay_cost + ext, ev, dynW

    def decide(self, sim, job):
        if job.deferrable and job.target is not None and not sim.full_reeval \
                and sim.t < job.target - sim.cp.tick_s - 1e-9:
            return ("defer", None)
        now = sim.t
        ci_ref = sim.ci_f.mean(now, now + 24 * HOUR)
        cands = sim.candidates(job)
        if not cands:
            # resource-blocked now; deferrable jobs still record a target for caching, but queue position holds
            job.target = now
            return ("blocked", None)
        running = list(sim.running.values())
        best_now, c_now, ev_now = None, math.inf, None
        for pl in cands:
            c, ev, dW = self._cost(sim, job, now, pl, running, ci_ref)
            if c < c_now - 1e-9:
                best_now, c_now, ev_now = pl, c, ev
        R_est = ev_now["R"]
        ts = _start_grid(sim, job, R_est)
        if len(ts) > 1:
            best_t, best_c = now, c_now
            for t in ts[1:]:
                free_t = sim.forecast_free(t)
                claimed = 0
                if self.claim_aware:
                    claimed = sum(j.n_gpus for j in sim.queue
                                  if j.jid < job.jid and j.target is not None and j.target <= t + 1e-9)
                if sum(free_t) - claimed < job.n_gpus:
                    continue
                active = sim.active_at(t)
                load_t = sim.slowdowns([(j.jid, j.pods_list, j.demand) for j in active])[1] if active else np.zeros(sim.cp.n_pods)
                cl = sim.candidates(job, free=free_t, load=load_t)
                if not cl:
                    continue
                for pl in cl:
                    c, ev, dW = self._cost(sim, job, float(t), pl, active, ci_ref)
                    if sim.cap_now() is not None:
                        extra_pods = [p for p in ev["pods"] if load_t[p] <= 1e-12] if ev["k"] > 1 else []
                        if sim.est_total_power(active, dW, ()) > sim.cap_now() + 1e-6:
                            continue
                    if c < best_c * (1.0 - self.defer_margin) - 1e-9:
                        best_t, best_c = float(t), c
            job.target = best_t
            if best_t > now + 1e-9:
                return ("defer", None)
        else:
            job.target = now
        return ("start", best_now)


class ForcedPolicy(Policy):
    """Executes a given decision table {jid: (start_time_s, 'compact'|'spread')}. Used by the exhaustive-search
    benchmark and by verification tests."""
    name = "FORCED"

    def __init__(self, decisions):
        self.dec = decisions

    def decide(self, sim, job):
        t0, mode = self.dec[job.jid]
        if sim.t < t0 - 1e-9:
            return ("defer", None)
        n = job.n_gpus
        P = sim.cp.n_pods
        if mode == "compact" and n <= sim.cp.gpus_per_pod:
            comp = [p for p in range(P) if sim.free[p] >= n]
            if comp:
                return ("start", {min(comp, key=lambda q: (sim.free[q], q)): n})
            return ("blocked", None)
        # spread over exactly two pods, as evenly as free GPUs allow
        order = sorted(range(P), key=lambda q: (-sim.free[q], q))
        a = n // 2
        if len(order) >= 2 and sim.free[order[0]] >= n - a and sim.free[order[1]] >= a:
            return ("start", {order[0]: n - a, order[1]: a})
        pl = sim.first_fit(n)
        if pl and len(pl) >= 2:
            return ("start", pl)
        return ("blocked", None)


POLICIES = {"P0": P0, "P2": P2, "P3": P3, "P4": P4}


# ----------------------------------------------------------------------------------------------
# One run + experiment design
# ----------------------------------------------------------------------------------------------


@dataclass
class Scenario:
    arrival: str = "poisson"
    util: float = 0.6
    ci: str = "synthetic_high_var"
    comm_scale: float = 1.0
    cap_frac: Optional[float] = None       # power cap as fraction of the cluster's maximum power
    fabric: str = "ocs"
    forecast_sigma: float = 0.0
    slack_lo: float = 2.0
    slack_hi: float = 12.0
    deferrable_frac: float = 0.7
    w_delay: float = 0.02
    tick_s: float = HOUR
    trace: str = ""                       # path to a trace CSV (arrival_s,n_gpus,duration_s); '' = synthetic arrivals


def build_world(sc: Scenario, rep: int, master_seed=MASTER_SEED_DEFAULT, profile_df: Optional[pd.DataFrame] = None,
                cluster_base: Optional[ClusterParams] = None, power_base: Optional[PowerParams] = None):
    cp = ClusterParams(**asdict(cluster_base)) if cluster_base else ClusterParams()
    pp = PowerParams(**asdict(power_base)) if power_base else PowerParams()
    apply_fabric(cp, pp, sc.fabric)
    cp.uplink_GBps *= cp.uplink_scale
    cp.tick_s = sc.tick_s
    if profile_df is None:
        profile_df = synthetic_profile(cp, comm_scale=sc.comm_scale)
    elif sc.comm_scale != 1.0:
        profile_df = profile_df.copy()
        profile_df["interpod_GB"] = profile_df["interpod_GB"] * sc.comm_scale
        if "comm_time_s" in profile_df:                       # communication time scales with volume
            profile_df["comm_time_s"] = profile_df["comm_time_s"] * sc.comm_scale
    prof = profile_dict(profile_df)
    if sc.cap_frac is not None:
        tmp = Sim([], prof, cp, pp, np.ones(10), P0())
        cp.power_cap_W = sc.cap_frac * tmp.max_power_W()
    if sc.trace:
        jobs, wmeta = workload_from_trace(pd.read_csv(sc.trace), rep, sc.util, cp, prof, master_seed=master_seed,
                                          deferrable_frac=sc.deferrable_frac, slack_range=(sc.slack_lo, sc.slack_hi))
    else:
        jobs, wmeta = make_workload(rep, sc.arrival, sc.util, cp, prof, master_seed=master_seed,
                                    deferrable_frac=sc.deferrable_frac, slack_range=(sc.slack_lo, sc.slack_hi))
    ci = make_ci(sc.ci, rep, int(cp.window_h) + 1, master_seed)
    cif = None
    if sc.forecast_sigma > 0:
        rf = rng_for(master_seed, rep, "forecast")
        cif = ci * np.exp(rf.normal(0.0, sc.forecast_sigma, size=len(ci)))
    return jobs, prof, cp, pp, ci, cif, wmeta


def run_one(sc: Scenario, policy_name: str, rep: int, master_seed=MASTER_SEED_DEFAULT,
            profile_df=None, cluster_base=None, power_base=None, check=False, record_trace=False):
    import time as _t
    t0 = _t.time()
    jobs, prof, cp, pp, ci, cif, wmeta = build_world(sc, rep, master_seed, profile_df, cluster_base, power_base)
    if policy_name == "P2raw":
        pol = P2(min_gain=0.0)
    elif policy_name.startswith("P4"):
        pol = P4(w_delay=sc.w_delay, claim_aware=(policy_name != "P4nc"))
    else:
        pol = POLICIES[policy_name]()
    sim = Sim(jobs, prof, cp, pp, ci, pol, ci_forecast=cif, check=check, record_trace=record_trace)
    m = sim.run()
    m.update(asdict(sc))
    m.update(policy=policy_name, rep=rep, workload_hash=workload_hash(jobs), runtime_s=_t.time() - t0,
             lam_per_h=wmeta["lam_per_h"], code_version=__version__,
             profile_source=(profile_df["source"].iloc[0] if profile_df is not None else "synthetic-parametric"))
    return (m, sim) if record_trace or check else m

In [ ]:
%%writefile dcsim_verify.py
"""
Verification battery for dcsim (textbook Ch. 13.2 programme): known-curve checks, degenerate cases, internal
consistency, distributional checks, discretisation insensitivity.  Each check returns a dict
{name, status in PASS|FAIL|WARN, detail}.  Run run_all() and keep the report with your results.
"""
from __future__ import annotations

import math
import time

import numpy as np
import pandas as pd
from scipy import stats

import dcsim
from dcsim import (HOUR, J_PER_KWH, ClusterParams, PowerParams, Job, Sim, P0, ForcedPolicy, Scenario,
                   rng_for, arrivals)


def _res(name, ok, detail, warn=False):
    status = "PASS" if ok else ("WARN" if warn else "FAIL")
    return dict(name=name, status=status, detail=detail)


def _tiny_profile(entries):
    return {k: v for k, v in entries.items()}


# ---------------------------------------------------------------------------------------------
def check_hand_case():
    """One 8-GPU compact job, constant CI: energy and carbon computed by hand."""
    cp, pp = ClusterParams(), PowerParams()
    cp.horizon_h, cp.window_h = 10, 10
    prof = {("hand", 8, "compact"): (2.0, 0.8, 0.0)}
    job = Job(0, 0.0, "hand", 8, 1800.0, False, 1e9, 3600.0)
    ci = np.full(11, 400.0)
    sim = Sim([job], prof, cp, pp, ci, P0())
    m = sim.run()
    pg = 8 * (100 + 550 * 0.8)
    floor = 1.3 * (128 * 100 + 16 * 800 + 4 * 400 + 400)
    during = 1.3 * (pg + 120 * 100 + 16 * 800 + 4 * 400 + 400)
    E_kwh = (floor * 10 * HOUR + (during - floor) * 3600.0) / J_PER_KWH
    ok = (abs(m["energy_kWh"] - E_kwh) < 1e-6 and abs(m["carbon_kg"] - E_kwh * 0.4) < 1e-6
          and abs(job.finish - 3600.0) < 1e-6)
    return _res("hand-computed single job (energy, carbon, runtime)", ok,
                f"sim E={m['energy_kWh']:.6f} kWh vs hand {E_kwh:.6f}; finish={job.finish:.3f}s (expect 3600)")



def check_hand_case_varying_ci():
    """Job spanning three different CI hours: carbon must be split exactly per hour (catches CI-timing bugs)."""
    cp, pp = ClusterParams(), PowerParams()
    cp.horizon_h, cp.window_h = 6, 6
    prof = {("hand", 8, "compact"): (2.0, 0.8, 0.0)}
    job = Job(0, 0.0, "hand", 8, 4500.0, False, 1e9, 9000.0)          # runs 0..9000 s = 2.5 h
    ci = np.array([100.0, 300.0, 500.0, 50.0, 50.0, 50.0, 50.0])
    m = Sim([job], prof, cp, pp, ci, P0()).run() if False else None
    sim = Sim([job], prof, cp, pp, ci, P0())
    m = sim.run()
    floor = 1.3 * (128 * 100 + 16 * 800 + 4 * 400 + 400)
    during = 1.3 * (8 * (100 + 550 * 0.8) + 120 * 100 + 16 * 800 + 4 * 400 + 400)
    kwh = lambda W, s: W * s / J_PER_KWH
    carbon = (kwh(floor, 6 * 3600) * 0 +
              sum(kwh(floor, 3600) * c for c in ci[:6]) +
              kwh(during - floor, 3600) * 100 + kwh(during - floor, 3600) * 300 + kwh(during - floor, 1800) * 500)
    ok = abs(m["carbon_kg"] * 1000 - carbon) < 1e-6 and abs(job.finish - 9000.0) < 1e-6
    return _res("hand-computed carbon with hourly-varying CI (job spans three CI hours)", ok,
                f"sim {m['carbon_kg'] * 1000:.4f} g vs hand {carbon:.4f} g")


def check_carbon_misaligned_tick():
    """Regression (audit finding 1): scheduler tick = 2 h while CI changes hourly; carbon must still integrate exactly."""
    cp, pp = ClusterParams(), PowerParams()
    cp.horizon_h, cp.window_h, cp.tick_s = 6, 6, 7200.0
    prof = {("hand", 8, "compact"): (2.0, 0.8, 0.0)}
    job = Job(0, 0.0, "hand", 8, 4500.0, False, 1e9, 9000.0)
    ci = np.array([100.0, 300.0, 500.0, 50.0, 50.0, 50.0, 50.0])
    sim = Sim([job], prof, cp, pp, ci, P0()); m = sim.run()
    floor = 1.3 * (128 * 100 + 16 * 800 + 4 * 400 + 400)
    during = 1.3 * (8 * (100 + 550 * 0.8) + 120 * 100 + 16 * 800 + 4 * 400 + 400)
    kwh = lambda W, s: W * s / J_PER_KWH
    carbon = (sum(kwh(floor, 3600) * c for c in ci[:6]) +
              kwh(during - floor, 3600) * 100 + kwh(during - floor, 3600) * 300 + kwh(during - floor, 1800) * 500)
    ok = abs(m["carbon_kg"] * 1000 - carbon) < 1e-6
    return _res("carbon integration exact with a 2-hour scheduler tick (CI changes hourly)", ok,
                f"sim {m['carbon_kg'] * 1000:.4f} g vs hand {carbon:.4f} g")


def check_k_extension():
    """Jobs over k > 2 pods: measured rows take precedence; otherwise the 2-pod row is extended by ClusterParams.k_rule.
    Checks the three rules against their formulas, the compact floor, and that 'hier' keeps the v0.1.0 numbers."""
    cp = ClusterParams(n_pods=4, gpus_per_pod=32)
    prof = {("x", 64, "compact"): (2.68, 2.0 / 2.68, 0.0, 0.68), ("x", 64, "spread"): (22.853, 2.0 / 22.853, 882.0, 20.853)}
    job = Job(0, 0.0, "x", 64, 10.0, False, 1e9, 3600.0)
    got, ok = [], True
    for rule, k, want_gb, want_comm in (("hier", 3, 882.0 * 4 / 3, 20.853 * 4 / 3), ("hier", 4, 882.0 * 1.5, 20.853 * 1.5),
                                        ("ring_interleaved", 3, 882.0 * 2 / 3, 20.853 * 2 / 3),
                                        ("ring_interleaved", 4, 441.0, 20.853 / 2), ("ring_aligned", 4, 882.0, 20.853)):
        c = ClusterParams(**{**dcsim.asdict(cp), "k_rule": rule})
        it, fc, gb, comm = Sim([], prof, c, PowerParams(), np.ones(10), P0()).prof(job, k)
        ok &= abs(gb - want_gb) < 1e-9 and abs(comm - want_comm) < 1e-9
        if rule == "hier":
            ok &= abs(it - 22.853) < 1e-12                       # v0.1.0: uncongested iteration time of the 2-pod row
        else:
            ok &= abs(it - (2.0 + want_comm)) < 1e-9 and abs(fc * it - 2.0) < 1e-9   # it = t_comp + comm (no overlap)
        got.append(f"{rule} k={k}: GB {gb:.1f} (want {want_gb:.1f})")
    # compact floor: a very spread interleaved job never communicates faster than the same job inside one pod
    c = ClusterParams(**{**dcsim.asdict(cp), "k_rule": "ring_interleaved", "n_pods": 64, "gpus_per_pod": 1})
    comm_floor = Sim([], prof, c, PowerParams(), np.ones(10), P0()).prof(job, 64)[3]
    ok &= abs(comm_floor - 0.68) < 1e-12
    # measured row for k=3 wins; k=4 then extends from the measured k=3 row
    prof3 = dict(prof); prof3[("x", 64, "spread_k3")] = (12.0, 2.0 / 12.0, 500.0, 10.0)
    c = ClusterParams(**{**dcsim.asdict(cp), "k_rule": "ring_interleaved"})
    sim = Sim([], prof3, c, PowerParams(), np.ones(10), P0())
    m3, m4 = sim.prof(job, 3), sim.prof(job, 4)
    ok &= m3 == (12.0, 2.0 / 12.0, 500.0, 10.0) and abs(m4[2] - 500.0 * (2 / 4) / (2 / 3)) < 1e-9
    return _res("jobs over k>2 pods: measured rows first, then the k_rule extension (hier / flat-ring), compact floor", ok,
                "; ".join(got) + f"; floor {comm_floor:.2f}s; measured k=3 used {m3[2]:.0f} GB, k=4 from it {m4[2]:.0f} GB")


def check_congestion_degenerate():
    """Spread job: runtime must equal n*it*(fc+(1-fc)*s) with s from the load ratio."""
    out = []
    for uplink, expect_s in [(40.0, 1.0), (10.0, 2.0)]:
        cp, pp = ClusterParams(n_pods=2, gpus_per_pod=32), PowerParams()
        cp.horizon_h, cp.window_h = 10, 10
        cp.uplink_GBps, cp.reconf_delay_s = uplink, 0.0
        prof = {("d", 16, "spread"): (1.0, 0.5, 20.0), ("d", 16, "compact"): (1.0, 0.5, 0.0)}
        job = Job(0, 0.0, "d", 16, 1000.0, False, 1e9, 1000.0)
        sim = Sim([job], prof, cp, pp, np.full(11, 300.0), ForcedPolicy({0: (0.0, "spread")}))
        sim.run()
        want = 1000 * 1.0 * (0.5 + 0.5 * expect_s)
        out.append((uplink, expect_s, job.finish - job.start, want))
    # two identical concurrent spread jobs share the same two uplinks: load doubles
    cp, pp = ClusterParams(n_pods=2, gpus_per_pod=32), PowerParams()
    cp.horizon_h, cp.window_h = 10, 10
    cp.uplink_GBps, cp.reconf_delay_s = 40.0, 0.0
    prof = {("d", 16, "spread"): (1.0, 0.5, 20.0), ("d", 16, "compact"): (1.0, 0.5, 0.0)}
    jobs = [Job(i, 0.0, "d", 16, 1000.0, False, 1e9, 1000.0) for i in range(2)]
    sim = Sim(jobs, prof, cp, pp, np.full(11, 300.0), ForcedPolicy({0: (0.0, "spread"), 1: (0.0, "spread")}))
    sim.run()
    want2 = 1000 * (0.5 + 0.5 * 1.0)  # load 40 on uplink 40 -> ratio 1 -> s=1
    out.append(("2 jobs, ratio 1", 1.0, jobs[1].finish - jobs[1].start, want2))
    cp.uplink_GBps = 20.0
    jobs = [Job(i, 0.0, "d", 16, 1000.0, False, 1e9, 1000.0) for i in range(2)]
    sim = Sim(jobs, prof, cp, pp, np.full(11, 300.0), ForcedPolicy({0: (0.0, "spread"), 1: (0.0, "spread")}))
    sim.run()
    out.append(("2 jobs, ratio 2", 2.0, jobs[1].finish - jobs[1].start, 1000 * (0.5 + 0.5 * 2.0)))
    # fully hidden communication (fc=1, exposed=0) must still slow down under congestion via comm_time_s
    cp, pp = ClusterParams(n_pods=2, gpus_per_pod=32), PowerParams()
    cp.horizon_h, cp.window_h = 10, 10
    cp.uplink_GBps, cp.reconf_delay_s = 10.0, 0.0
    prof = {("h", 16, "spread"): (1.0, 1.0, 20.0, 0.4), ("h", 16, "compact"): (1.0, 1.0, 0.0, 0.0)}
    job = Job(0, 0.0, "h", 16, 1000.0, False, 1e9, 1000.0)
    Sim([job], prof, cp, pp, np.full(11, 300.0), ForcedPolicy({0: (0.0, "spread")})).run()
    out.append(("hidden comm, s=2", 2.0, job.finish - job.start, 1000 * (1.0 + 0.4 * (2.0 - 1.0))))
    ok = all(abs(a - b) < 1e-6 for *_, a, b in out)
    return _res("congestion degenerate cases (runtime = n*(it + comm*(s-1)), incl. fully hidden communication)", ok,
                "; ".join(f"{x[0]}: got {x[2]:.3f}s want {x[3]:.3f}s" for x in out))


def erlang_c_W(lam, mu, c):
    a = lam / mu
    s = sum(a ** k / math.factorial(k) for k in range(c))
    top = a ** c / math.factorial(c) * c / (c - a)
    Cw = top / (s + top)
    return Cw / (c * mu - lam) + 1.0 / mu


def _mmc_run(rho, c, mu, rep, horizon_h=200.0, master=dcsim.MASTER_SEED_DEFAULT):
    cp, pp = ClusterParams(n_pods=1, gpus_per_pod=c, gpus_per_server=1), PowerParams()
    cp.horizon_h, cp.window_h = horizon_h, horizon_h + 60.0
    lam = rho * c * mu
    ra, rj = rng_for(master, rep, "arrival"), rng_for(master, rep, "attrs")
    times = arrivals("poisson", lam, horizon_h * HOUR, ra)
    prof = {("m", 1, "compact"): (1.0, 1.0, 0.0)}
    jobs = [Job(i, float(t), "m", 1, float(-math.log1p(-rj.random()) / mu), False, 1e18, 1.0)
            for i, t in enumerate(times)]
    sim = Sim(jobs, prof, cp, pp, np.full(int(cp.window_h) + 2, 100.0), P0(), check=False)
    sim.run()
    return jobs, sim, lam, cp


def check_mmc(rhos=(0.5, 0.8), c=8, mu=1.0 / 600.0, reps=8):
    """DES verification against Erlang-C (textbook Ch. 9.3): terminating runs, warm-up deleted, replications."""
    details, ok_all = [], True
    for rho in rhos:
        means = []
        for r in range(reps):
            jobs, sim, lam, cp = _mmc_run(rho, c, mu, r)
            warm = 0.1 * cp.horizon_h * HOUR
            soj = [j.finish - j.arrival for j in jobs if j.finish is not None and j.arrival > warm]
            means.append(np.mean(soj))
        m, s = np.mean(means), np.std(means, ddof=1)
        half = stats.t.ppf(0.995, reps - 1) * s / math.sqrt(reps)     # 99% interval (fewer false alarms)
        theory = erlang_c_W(rho * c * mu, mu, c)
        ok = abs(m - theory) <= half or abs(m - theory) / theory < 0.03
        ok_all &= ok
        details.append(f"rho={rho}: sim W={m:.1f}s ±{half:.1f} (99% CI) vs Erlang-C {theory:.1f}s")
    return _res("M/M/c mean sojourn vs Erlang-C (replications, warm-up deleted)", ok_all, " | ".join(details))


def check_little():
    """Internal consistency: time-average number in system L must equal lambda_hat * W_hat (identity)."""
    jobs, sim, lam, cp = _mmc_run(0.8, 8, 1 / 600.0, 0, horizon_h=100.0)
    W = cp.window_h * HOUR
    L = sim.area_n / W
    lam_hat = len(jobs) / W
    What = np.mean([j.finish - j.arrival for j in jobs])
    ok = all(j.finish is not None for j in jobs) and abs(L - lam_hat * What) / L < 1e-9
    return _res("Little's law L = lambda*W (time average vs customer average)", ok,
                f"L={L:.6f}, lambda*W={lam_hat * What:.6f}")


def check_energy_conservation():
    sc = Scenario(util=0.6, comm_scale=4.0, arrival="bursty")
    m, sim = dcsim.run_one(sc, "P0", 0, record_trace=True)
    tr = sim.trace
    E = sum((t1 - t0) * P for t0, t1, P, *_ in tr)
    ok1 = abs(E / J_PER_KWH - m["energy_kWh"]) / m["energy_kWh"] < 1e-9
    pue = sim.pp.PUE
    ok2 = all(abs(pue * (a + b + c + d) - P) < 1e-6 * max(P, 1) for _, _, P, a, b, c, d in tr)
    ok3 = abs(tr[0][0]) < 1e-9 and abs(tr[-1][1] - sim.W) < 1e-6 and all(abs(tr[i][1] - tr[i + 1][0]) < 1e-9 for i in range(len(tr) - 1))
    carb = sum((t1 - t0) * P / J_PER_KWH * sim.ci.at(t0) for t0, t1, P, *_ in tr)
    ok4 = abs(carb / 1000 - m["carbon_kg"]) / m["carbon_kg"] < 1e-9
    return _res("energy/carbon conservation (trace integral, component sum, contiguous cover)",
                ok1 and ok2 and ok3 and ok4, f"energy {ok1}, components {ok2}, coverage {ok3}, carbon {ok4}")


def check_invariants():
    bad = []
    for pol in ["P0", "P2", "P3", "P4", "P4nc"]:
        for sc in [Scenario(util=0.7, comm_scale=8.0, arrival="bursty"), Scenario(util=0.5, cap_frac=0.7)]:
            try:
                dcsim.run_one(sc, pol, 1, check=True)
            except AssertionError as e:
                bad.append(f"{pol}: {e}")
    return _res("invariants every event (GPU conservation, no job in two states) for all policies", not bad,
                "ok" if not bad else "; ".join(bad))


def check_power_cap():
    sc = Scenario(util=0.7, cap_frac=0.62, ci="synthetic_high_var")
    res = []
    ok = True
    for pol in ["P0", "P2", "P3", "P4"]:
        m, sim = dcsim.run_one(sc, pol, 0, check=True)
        cap = sim.cp.power_cap_W
        ok &= (m["peak_kW"] * 1000 <= cap + 1e-3 and m["cap_exceed_h"] == 0.0)
        res.append(f"{pol}: peak {m['peak_kW']:.1f} kW <= cap {cap / 1000:.1f} kW, blocked {m['cap_block_events']}")
    return _res("power cap is never exceeded (admission control)", ok, " | ".join(res))


def check_crn():
    sc = Scenario(util=0.6)
    h = {pol: dcsim.run_one(sc, pol, 3)["workload_hash"] for pol in ["P0", "P2", "P3", "P4"]}
    same = len(set(h.values())) == 1
    h_other = dcsim.run_one(sc, "P0", 4)["workload_hash"]
    h_rep = dcsim.run_one(sc, "P0", 3)["workload_hash"]
    ok = same and h_other != h["P0"] and h_rep == h["P0"]
    return _res("common random numbers: identical job stream across policies, distinct across replications, repeatable",
                ok, f"hashes {sorted(set(h.values()))[0]} (policies), {h_other} (other rep)")


def check_generators():
    cp = ClusterParams()
    r = []
    ok = True
    lam = 20.0 / HOUR
    # Poisson gaps ~ Exp(lam): KS test
    t = arrivals("poisson", lam, 4000 * HOUR, rng_for(1, 0, "arrival"))
    gaps = np.diff(t)
    ks = stats.kstest(gaps * lam, "expon")
    ok &= ks.pvalue > 0.001
    r.append(f"poisson gaps KS p={ks.pvalue:.3f}")
    # long-run rates
    for model in ["poisson", "bursty", "diurnal"]:
        tt = arrivals(model, lam, 4000 * HOUR, rng_for(1, 1, "arrival"))
        rate = len(tt) / 4000
        ok &= abs(rate - 20.0) / 20.0 < 0.05
        r.append(f"{model} rate {rate:.2f}/h (target 20)")
    tt = arrivals("diurnal", lam, 4000 * HOUR, rng_for(1, 2, "arrival"))
    hod = ((tt / HOUR) % 24).astype(int)
    cnt = np.bincount(hod, minlength=24)
    ratio = cnt[13:16].mean() / cnt[1:4].mean()
    ok &= 4.0 < ratio < 20.0
    r.append(f"diurnal peak/trough ≈ {ratio:.1f} (theory 9)")
    # bursty index of dispersion > 1
    tb = arrivals("bursty", lam, 4000 * HOUR, rng_for(1, 3, "arrival"))
    cb = np.bincount((tb // HOUR).astype(int), minlength=4000)
    iod = cb.var() / cb.mean()
    tp = arrivals("poisson", lam, 4000 * HOUR, rng_for(1, 3, "arrival"))
    cpois = np.bincount((tp // HOUR).astype(int), minlength=4000)
    ok &= iod > 1.5 * cpois.var() / cpois.mean()
    r.append(f"hourly index of dispersion bursty {iod:.1f} vs poisson {cpois.var() / cpois.mean():.2f}")
    # CI scenarios: variability ordering
    lo = dcsim.ci_variability(dcsim.make_ci("synthetic_low_var", 0, 24 * 30))
    hi = dcsim.ci_variability(dcsim.make_ci("synthetic_high_var", 0, 24 * 30))
    ok &= hi["daily_range_over_mean"] > 3 * lo["daily_range_over_mean"]
    r.append(f"CI daily-range/mean low {lo['daily_range_over_mean']:.2f} vs high {hi['daily_range_over_mean']:.2f}")
    return _res("generator distributional checks (KS, long-run rates, burstiness, CI variability)", ok, " | ".join(r))


def check_trace_path():
    """Trace-driven workload: identical across policies, different window per replication, every job finishes."""
    import tempfile, os
    rng = np.random.default_rng(5)
    n = 3000
    df = pd.DataFrame(dict(arrival_s=np.cumsum(rng.exponential(300, n)),
                           n_gpus=rng.choice([1, 2, 4, 8, 16, 32, 64], n, p=[.1, .1, .1, .3, .2, .15, .05]),
                           duration_s=np.exp(rng.normal(np.log(3600), 1.0, n))))
    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, "trace.csv")
        df.to_csv(p, index=False)
        sc = Scenario(util=0.6, trace=p)
        hs = {pol: dcsim.run_one(sc, pol, 0) for pol in ["P0", "P2", "P4"]}
        h1 = dcsim.run_one(sc, "P0", 1)["workload_hash"]
    same = len({m["workload_hash"] for m in hs.values()}) == 1
    done = all(m["n_incomplete"] == 0 for m in hs.values())
    ok = same and done and h1 != hs["P0"]["workload_hash"]
    return _res("trace-driven workload (SYNTHETIC test trace): CRN across policies, window per replication, all jobs finish",
                ok, f"same stream {same}, all complete {done}, rep1 differs {h1 != hs['P0']['workload_hash']}")


def check_rng_streams():
    a = rng_for(7, 0, "arrival").random(5)
    b = rng_for(7, 0, "attrs").random(5)
    c = rng_for(7, 1, "arrival").random(5)
    d = rng_for(7, 0, "arrival").random(5)
    ok = (not np.allclose(a, b)) and (not np.allclose(a, c)) and np.allclose(a, d)
    return _res("RNG streams: separated by stream and replication, reproducible (SeedSequence)", ok, "ok" if ok else "stream clash")


def check_tick_insensitivity(reps=3, tol=0.02):
    """Discretisation insensitivity: halving the scheduler tick should not move the carbon result (WARN, not FAIL)."""
    diffs = []
    for pol in ["P2", "P4"]:
        a = [dcsim.run_one(Scenario(util=0.6, tick_s=3600.0), pol, r)["carbon_dyn_kg"] for r in range(reps)]
        b = [dcsim.run_one(Scenario(util=0.6, tick_s=1800.0), pol, r)["carbon_dyn_kg"] for r in range(reps)]
        diffs.append(abs(np.mean(a) - np.mean(b)) / np.mean(a))
    ok = max(diffs) < tol
    return _res("scheduler tick halved (3600s -> 1800s): carbon unchanged within tolerance", ok,
                f"relative change P2 {diffs[0]:.3%}, P4 {diffs[1]:.3%} (tolerance {tol:.0%})", warn=True)


# ---------------------------------------------------------------------------------------------
# Calibration vs validation of the GPU power model (never validate on the calibration data)
# ---------------------------------------------------------------------------------------------

def calibrate_gpu_power(df: pd.DataFrame, holdout_workloads=None):
    """df columns: workload, frac_compute (0..1), power_w (per GPU).  Model: P = P_idle + (P_max-P_idle)*fc.
    Fit on training workloads, report error on HELD-OUT workloads (leave-one-workload-out if holdout is None)."""
    def fit(d):
        A = np.vstack([np.ones(len(d)), d["frac_compute"].to_numpy()]).T
        coef, *_ = np.linalg.lstsq(A, d["power_w"].to_numpy(), rcond=None)
        return coef[0], coef[0] + coef[1]
    ws = sorted(df["workload"].unique())
    if len(ws) < 3:
        raise ValueError("need at least 3 distinct workloads to calibrate and validate separately")
    rows = []
    for w in (holdout_workloads or ws):
        tr, te = df[df.workload != w], df[df.workload == w]
        pi, pm = fit(tr)
        pred = pi + (pm - pi) * te["frac_compute"].to_numpy()
        err = pred - te["power_w"].to_numpy()
        rows.append(dict(heldout=w, P_idle=pi, P_max=pm, mape=float(np.mean(np.abs(err) / te["power_w"].to_numpy())),
                         rmse_w=float(np.sqrt(np.mean(err ** 2)))))
    pi, pm = fit(df)
    return dict(P_idle_all=pi, P_max_all=pm, validation=pd.DataFrame(rows))


def check_calibration_selftest():
    rng = np.random.default_rng(0)
    rows = []
    for w, lo, hi in [("a", 0.5, 0.95), ("b", 0.6, 1.0), ("c", 0.4, 0.9), ("d", 0.7, 1.0)]:
        fc = rng.uniform(lo, hi, 12)
        rows += [dict(workload=w, frac_compute=f, power_w=90 + 560 * f + rng.normal(0, 8)) for f in fc]
    out = calibrate_gpu_power(pd.DataFrame(rows))
    ok = abs(out["P_idle_all"] - 90) < 30 and abs(out["P_max_all"] - 650) < 30 and out["validation"]["mape"].max() < 0.05
    return _res("calibrate/validate split recovers known parameters on SYNTHETIC data (self-test only)", ok,
                f"fit P_idle={out['P_idle_all']:.0f} W, P_max={out['P_max_all']:.0f} W (truth 90, 650); "
                f"worst held-out MAPE {out['validation']['mape'].max():.2%}")


def run_all(quick=False):
    checks = [check_hand_case, check_hand_case_varying_ci, check_carbon_misaligned_tick, check_congestion_degenerate, check_little, check_energy_conservation,
              check_invariants, check_power_cap, check_crn, check_trace_path, check_rng_streams, check_generators,
              check_calibration_selftest, check_k_extension]
    if not quick:
        checks += [check_mmc, check_tick_insensitivity]
    rep = []
    for fn in checks:
        t0 = time.time()
        try:
            r = fn()
        except Exception as e:                      # a crash is a failure, report it
            r = _res(fn.__name__, False, f"EXCEPTION {type(e).__name__}: {e}")
        r["seconds"] = round(time.time() - t0, 1)
        rep.append(r)
    return rep


def print_report(rep):
    for r in rep:
        print(f"[{r['status']:4s}] {r['name']}  ({r['seconds']}s)\n         {r['detail']}")
    n_fail = sum(r["status"] == "FAIL" for r in rep)
    n_warn = sum(r["status"] == "WARN" for r in rep)
    print(f"\n{len(rep)} checks: {len(rep) - n_fail - n_warn} PASS, {n_warn} WARN, {n_fail} FAIL")
    return n_fail == 0

In [ ]:
%%writefile dcsim_exp.py
"""
Experiment layer for dcsim: designed experiments (not wandering), checkpointed parallel runner, paired CRN analysis
with t-intervals, an exhaustive-search benchmark for the P4 heuristic, and plots.
"""
from __future__ import annotations

import itertools
import json
import math
import os
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict

import numpy as np
import pandas as pd
from scipy import stats

import dcsim
from dcsim import HOUR, Scenario, ClusterParams, PowerParams, Job, Sim

SC_COLS = list(asdict(Scenario()).keys())
KEY_COLS = ["tag"] + SC_COLS
POLICIES_MAIN = ["P0", "P2", "P3", "P4"]
POLICIES_SENS = ["P0", "P2", "P2raw", "P3", "P4", "P4nc"]
METRICS = ["carbon_dyn_kg", "carbon_kg", "energy_dyn_kWh", "jct_mean_h", "jct_p95_h", "wait_mean_h",
           "deadline_viol_frac", "peak_kW", "net_inflation", "spread_frac", "ci_at_start_gpuw",
           "util_time_avg", "max_spread_running", "cap_exceed_h", "gpuh_frac_k3plus"]


# ----------------------------------------------------------------------------------------------
# Designs
# ----------------------------------------------------------------------------------------------

COMM_MAIN, COMM_SWEEP, BASE_COMM = (1.0, 8.0), [1.0, 2.0, 4.0, 6.0, 8.0, 12.0], 6.0   # synthetic-profile defaults


UTIL_MAIN, UTIL_SWEEP, BASE_UTIL = (0.5, 0.8), [0.4, 0.6, 0.7, 0.8, 0.9], 0.7


def set_stress_levels(on: bool):
    """Worst-case (interleaved) profiles make spread jobs 1.4-8.5x slower, so the nominal utilisation (defined on COMPACT
    run times) is far above real capacity and runs cannot drain. Stress mode lowers the load and communication levels so the
    cluster is loaded but not collapsed; the interleaved profile is then reported as a bound, not as the baseline."""
    global UTIL_MAIN, UTIL_SWEEP, BASE_UTIL, COMM_MAIN, COMM_SWEEP, BASE_COMM
    if on:
        UTIL_MAIN, UTIL_SWEEP, BASE_UTIL = (0.2, 0.3), [0.1, 0.2, 0.3, 0.4, 0.5], 0.3
        COMM_MAIN, COMM_SWEEP, BASE_COMM = (0.5, 1.0), [0.25, 0.5, 0.75, 1.0, 1.5, 2.0], 0.5
    else:
        UTIL_MAIN, UTIL_SWEEP, BASE_UTIL = (0.5, 0.8), [0.4, 0.6, 0.7, 0.8, 0.9], 0.7


def set_comm_levels(measured_profile: bool):
    """comm_scale multiplies the communication volume AND time of the profile table. With the synthetic placeholder,
    1.0 is a gentle level and the network effect appears around 6-8. With a MEASURED (ASTRA-sim) table, 1.0 is already the
    measured collective cost, so useful levels are 0.5-4 (a larger model / gradient) and 8x would be unrealistic."""
    global COMM_MAIN, COMM_SWEEP, BASE_COMM
    if measured_profile:
        COMM_MAIN, COMM_SWEEP, BASE_COMM = (1.0, 3.0), [0.5, 1.0, 1.5, 2.0, 3.0, 4.0], 1.0
    else:
        COMM_MAIN, COMM_SWEEP, BASE_COMM = (1.0, 8.0), [1.0, 2.0, 4.0, 6.0, 8.0, 12.0], 6.0


def main_design(R=10, arrivals=("poisson", "bursty"), utils=None, comms=None,
                cis=("synthetic_low_var", "synthetic_high_var"), policies=POLICIES_MAIN, **fixed):
    """Full factorial over a few levels per factor, replicated R times (common random numbers across policies)."""
    comms = comms or COMM_MAIN
    utils = utils or UTIL_MAIN
    rows = []
    for ar, u, cs, ci in itertools.product(arrivals, utils, comms, cis):
        sc = dict(arrival=ar, util=u, comm_scale=cs, ci=ci, **fixed)
        for pol in policies:
            for r in range(R):
                rows.append(dict(tag="main", scenario=sc, policy=pol, rep=r))
    return rows


def base_scenario(**kw):
    d = dict(arrival="poisson", util=BASE_UTIL, comm_scale=BASE_COMM, ci="synthetic_high_var")
    d.update(kw)
    return d


def trace_design(trace_csv, R=10, utils=None, cis=("synthetic_high_var",), comms=None,
                 policies=POLICIES_MAIN):
    """Trace-driven arrivals (replication = different window of the trace). Communication intensity is ASSIGNED."""
    comms = comms or COMM_MAIN
    utils = utils or UTIL_MAIN
    rows = []
    for u, ci, cs in itertools.product(utils, cis, comms):
        sc = dict(trace=trace_csv, util=u, ci=ci, comm_scale=cs)
        for pol in policies:
            for r in range(R):
                rows.append(dict(tag="trace", scenario=sc, policy=pol, rep=r))
    return rows


def sensitivity_design(R=10, policies=POLICIES_SENS, ci_extra=None):
    """One-factor-at-a-time sweeps around a base cell. OAT cannot see interactions: the main factorial covers those."""
    sweeps = {
        "comm_scale": list(COMM_SWEEP),
        "util": list(UTIL_SWEEP),
        "cap_frac": [None, 0.8, 0.7, 0.62],
        "forecast_sigma": [0.0, 0.1, 0.3],
        "fabric": ["ocs", "electrical"],
        "slack": [(1.0, 4.0), (2.0, 12.0), (4.0, 24.0)],
        "w_delay": [0.0, 0.02, 0.1],
        "deferrable_frac": [0.3, 0.7, 1.0],
        "arrival": ["poisson", "bursty", "diurnal"],
    }
    if ci_extra:
        sweeps["ci"] = ["synthetic_low_var", "synthetic_high_var"] + list(ci_extra)
    rows = []
    for factor, levels in sweeps.items():
        for lv in levels:
            sc = base_scenario()
            if factor == "slack":
                sc["slack_lo"], sc["slack_hi"] = lv
            else:
                sc[factor] = lv
            for pol in policies:
                for r in range(R):
                    rows.append(dict(tag=f"sens:{factor}", scenario=sc, policy=pol, rep=r))
    return rows


def _key(item):
    full = asdict(Scenario(**item["scenario"]))
    return json.dumps([item["tag"], full, item["policy"], item["rep"]], sort_keys=True, default=str)


def _work(args):
    item, profile_df, master_seed, cluster_base, power_base = args
    sc = Scenario(**item["scenario"])
    try:
        m = dcsim.run_one(sc, item["policy"], item["rep"], master_seed=master_seed, profile_df=profile_df,
                          cluster_base=cluster_base, power_base=power_base)
        m["tag"] = item["tag"]
        m["key"] = _key(item)
        m["error"] = ""
    except Exception as e:                      # keep the sweep alive, record the failure
        m = dict(tag=item["tag"], key=_key(item), policy=item["policy"], rep=item["rep"],
                 error=f"{type(e).__name__}: {e}", **asdict(Scenario(**item["scenario"])))
    return m


def run_design(design, out_csv, workers=None, profile_df=None, master_seed=dcsim.MASTER_SEED_DEFAULT,
               cluster_base=None, power_base=None, batch=200, verbose=True):
    """Checkpointed: results append to out_csv; re-running skips finished keys (survives Colab disconnects)."""
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv, usecols=["key"])["key"])
    todo = [it for it in design if _key(it) not in done]
    if verbose:
        print(f"{len(design)} runs in design, {len(design) - len(todo)} already done, {len(todo)} to run")
    workers = workers or max(1, (os.cpu_count() or 1))
    t0 = time.time()
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        args = [(it, profile_df, master_seed, cluster_base, power_base) for it in chunk]
        if workers > 1:
            with ProcessPoolExecutor(max_workers=workers) as ex:
                res = list(ex.map(_work, args, chunksize=4))
        else:
            res = [_work(a) for a in args]
        df = pd.DataFrame(res)
        df.to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
        if verbose:
            el = time.time() - t0
            print(f"  {min(i + batch, len(todo))}/{len(todo)} done, {el:.0f}s elapsed")
    return out_csv


# ----------------------------------------------------------------------------------------------
# Analysis (replications + t-intervals; paired differences under common random numbers)
# ----------------------------------------------------------------------------------------------

def load_runs(csv):
    df = pd.read_csv(csv)
    if "error" in df and (df["error"].fillna("") != "").any():
        print("WARNING: failed runs:", int((df["error"].fillna("") != "").sum()))
    df = df[df["error"].fillna("") == ""].copy()
    df["cap_frac"] = df["cap_frac"].astype(float).fillna(-1.0)     # -1 = no power cap
    if "trace" in df:
        df["trace"] = df["trace"].fillna("").astype(str)             # '' = synthetic arrivals (NaN would break joins)
    return df


def _cell_cols(df):
    return [c for c in KEY_COLS if c in df.columns]


def tci(x, level=0.95):
    x = np.asarray(x, float)
    n = len(x)
    if n < 2:
        return float("nan")
    return float(stats.t.ppf(0.5 + level / 2, n - 1) * x.std(ddof=1) / math.sqrt(n))


def summarize(df, metrics=("carbon_dyn_kg", "jct_mean_h", "deadline_viol_frac", "net_inflation", "spread_frac"),
              by=None):
    by = by or (_cell_cols(df) + ["policy"])
    rows = []
    for key, g in df.groupby(by, dropna=False):
        row = dict(zip(by, key if isinstance(key, tuple) else (key,)))
        row["n"] = len(g)
        row["censored_runs"] = int((g["n_incomplete"] > 0).sum())
        for m in metrics:
            row[m] = g[m].mean()
            row[m + "_ci"] = tci(g[m])
        rows.append(row)
    return pd.DataFrame(rows)


def paired(df, a, b, metric="carbon_dyn_kg", relative=True, by=None, drop_censored=True):
    """Paired difference a - b over replications with identical job streams (CRN). Returns per-cell mean,
    95% t-interval half-width, whether the interval excludes zero, and how many replications were censored
    (runs where some job did not finish inside the accounting window are dropped from paired comparisons)."""
    by = by or _cell_cols(df)
    keep = by + ["rep", "policy", metric, "n_incomplete", "workload_hash"]
    d = df[df.policy.isin([a, b])][keep]
    pa = d[d.policy == a].set_index(by + ["rep"])
    pb = d[d.policy == b].set_index(by + ["rep"])
    j = pa.join(pb, lsuffix="_a", rsuffix="_b", how="inner")
    assert (j["workload_hash_a"] == j["workload_hash_b"]).all(), "CRN violated: job streams differ between policies"
    cens = (j["n_incomplete_a"] > 0) | (j["n_incomplete_b"] > 0)
    j["censored"] = cens
    if drop_censored:
        jj = j[~cens]
    else:
        jj = j
    diff = jj[metric + "_a"] - jj[metric + "_b"]
    if relative:
        diff = 100.0 * diff / jj[metric + "_b"].abs().replace(0, np.nan)
    out = diff.groupby(level=list(range(len(by)))).agg(["mean", "count"])
    out["ci95"] = diff.groupby(level=list(range(len(by)))).apply(lambda x: tci(x.values))
    out["censored_reps"] = j["censored"].groupby(level=list(range(len(by)))).sum()
    out = out.rename(columns={"mean": "diff", "count": "n"}).reset_index()
    out["excludes_zero"] = (out["diff"].abs() > out["ci95"]) & out["ci95"].notna()
    out["a"], out["b"], out["metric"], out["unit"] = a, b, metric, "%" if relative else "abs"
    return out


# ----------------------------------------------------------------------------------------------
# Exhaustive-search benchmark (small instance): how far is the P4 heuristic from the best achievable?
# ----------------------------------------------------------------------------------------------

def tiny_instance(comm_scale=12.0, ci_scenario="synthetic_high_var", rep=0):
    cp, pp = ClusterParams(n_pods=2, gpus_per_pod=32), PowerParams()
    cp.horizon_h, cp.window_h = 12.0, 60.0
    prof_df = dcsim.synthetic_profile(cp, comm_scale=comm_scale)
    prof = dcsim.profile_dict(prof_df)
    spec = [("llm_mid", 16, 0.2, 1.5), ("llm_mid", 16, 0.5, 1.2), ("llm_large", 32, 1.0, 1.0),
            ("llm_mid", 16, 1.5, 1.4), ("vision", 8, 2.0, 1.0)]
    jobs = []
    for i, (jt, n, arr_h, dur_h) in enumerate(spec):
        it = prof[(jt, n, "compact" if n <= cp.gpus_per_pod else "spread")][0]
        jobs.append(Job(i, arr_h * HOUR, jt, n, dur_h * HOUR / it, True, arr_h * HOUR + dur_h * HOUR * 8.0, dur_h * HOUR))
    ci = dcsim.make_ci(ci_scenario, rep, int(cp.window_h) + 1)
    return jobs, prof, cp, pp, ci


def objective(sim, jobs, w_delay, ci_ref):
    """Same scale as the P4 cost: dynamic carbon + w_delay * delay price."""
    pp = sim.pp
    carbon_dyn_g = sim.carbon_g - sim.idle_floor_W() * float(sim.ci.integral(0.0, sim.W)) / dcsim.J_PER_KWH
    price_kW = pp.PUE * pp.P_gpu_max / 1000.0
    delay = sum(j.n_gpus * price_kW * ci_ref * (j.finish - j.arrival) / HOUR for j in jobs if j.finish is not None)
    unfinished = sum(1 for j in jobs if j.finish is None)
    viol = sum(1 for j in jobs if j.finish is not None and j.finish > j.deadline) + unfinished
    return carbon_dyn_g + w_delay * delay, carbon_dyn_g, viol, unfinished


def _fresh_jobs(jobs):
    return [Job(j.jid, j.arrival, j.jtype, j.n_gpus, j.n_iters, j.deferrable, j.deadline, j.dur_ref_s) for j in jobs]


def oracle_benchmark(comm_scale=12.0, ci_scenario="synthetic_high_var", w_delay=0.02, max_delay_h=14,
                     restarts=300, seed=1, verbose=True):
    """Best-known-solution benchmark on a 5-job instance. Decision per job = (start hour, compact|spread).
    Search = coordinate descent from random restarts and from each heuristic's own decisions. This is NOT a proof of
    optimality (the space is too large to enumerate); it is a strong reference ("best known") against which the
    heuristics' gap is reported. The heuristics are included in the best-known set so the gap is never negative."""
    jobs0, prof, cp, pp, ci = tiny_instance(comm_scale, ci_scenario)
    ci_ref = float(np.mean(ci[:24]))

    def run(dec_or_policy, jobs):
        pol = dec_or_policy if not isinstance(dec_or_policy, dict) else dcsim.ForcedPolicy(dec_or_policy)
        sim = Sim(jobs, prof, ClusterParams(**asdict(cp)), pp, ci, pol)
        sim.run()
        return objective(sim, jobs, w_delay, ci_ref)

    res, seeds = {}, []
    for name in ["P0", "P2", "P3", "P4"]:
        jobs = _fresh_jobs(jobs0)
        pol = dcsim.P4(w_delay=w_delay) if name == "P4" else dcsim.POLICIES[name]()
        J, c, v, u = run(pol, jobs)
        res[name] = dict(J=J, carbon_dyn_kg=c / 1000, violations=v, unfinished=u)
        seeds.append({j.jid: (j.start if j.start is not None else j.arrival, "compact" if j.k == 1 else "spread")
                      for j in jobs})
    options = []
    for j in jobs0:
        base = math.ceil(j.arrival / HOUR) * HOUR
        o = [(base + d * HOUR, m) for d in range(max_delay_h + 1) for m in ("compact", "spread")
             if base + d * HOUR <= j.deadline - j.dur_ref_s * 1.2]
        options.append(o)

    def evalJ(dec):
        J, c, v, u = run(dec, _fresh_jobs(jobs0))
        return (J if (v == 0 and u == 0) else J + 1e9), c, v, u

    rng = dcsim.rng_for(seed, 0, "misc")
    starts = [dict(s) for s in seeds]
    for _ in range(restarts):
        starts.append({j.jid: options[i][int(rng.integers(len(options[i])))] for i, j in enumerate(jobs0)})
    best_J, best_dec, best_c, n_eval = math.inf, None, None, 0
    for dec in starts:
        cur = dict(dec)
        curJ = evalJ(cur)[0]
        n_eval += 1
        improved = True
        while improved:
            improved = False
            for i, j in enumerate(jobs0):
                for opt in options[i]:
                    if opt == cur[j.jid]:
                        continue
                    trial = dict(cur)
                    trial[j.jid] = opt
                    Jt = evalJ(trial)[0]
                    n_eval += 1
                    if Jt < curJ - 1e-9:
                        cur, curJ, improved = trial, Jt, True
        if curJ < best_J:
            best_J, best_dec = curJ, cur
    Jb, cb, vb, ub = evalJ(best_dec)
    ok_heur = {k: v["J"] for k, v in res.items() if v["violations"] == 0 and v["unfinished"] == 0}
    ref = min([best_J] + list(ok_heur.values()))
    res["BEST_KNOWN"] = dict(J=ref, carbon_dyn_kg=cb / 1000 if ref == best_J else float("nan"),
                             violations=vb if ref == best_J else 0, unfinished=ub if ref == best_J else 0)
    df = pd.DataFrame(res).T
    df["gap_vs_best_known_%"] = 100 * (df["J"] - ref) / ref
    if verbose:
        print(f"coordinate-descent search: {len(starts)} starts, {n_eval} simulations; "
              f"local-search best J={best_J:.0f}")
    return df, best_dec


# ----------------------------------------------------------------------------------------------
# Plots (palette: reference categorical slots 1-4, validated; policies keep their colour everywhere)
# ----------------------------------------------------------------------------------------------

POLICY_COLOR = {"P0": "#6b6a66", "P2": "#2a78d6", "P3": "#eb6834", "P4": "#1baf7a", "P4nc": "#eda100", "P2raw": "#8a63c9"}
POLICY_MARK = {"P0": "o", "P2": "s", "P3": "^", "P4": "D", "P4nc": "v", "P2raw": "P"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"


def _style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#b8b7b2")
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(INK2)
    ax.yaxis.label.set_color(INK2)


def plot_onset(df_sens, factor="comm_scale", metric="carbon_dyn_kg", ylabel=None, outfile=None):
    """Policy lines against one swept factor, with 95% intervals; direct labels at line ends + legend."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df_sens[df_sens.tag == f"sens:{factor}"]
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for pol, g in d.groupby("policy"):
        s = g.groupby(factor)[metric].agg(["mean", lambda x: tci(x)]).reset_index()
        s.columns = [factor, "mean", "ci"]
        ax.errorbar(s[factor], s["mean"], yerr=s["ci"], color=POLICY_COLOR[pol], marker=POLICY_MARK[pol], ms=6,
                    lw=1.8, capsize=0, elinewidth=0.8, label=pol, mec="#fcfcfb", mew=1.5)
    ax.set_xlabel(factor)
    ax.set_ylabel(ylabel or metric)
    ax.legend(frameon=False, fontsize=9, loc="center left", bbox_to_anchor=(1.01, 0.5))
    _style(ax)
    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, dpi=160, facecolor="#fcfcfb")
    return fig


def plot_paired_bars(tab, title, outfile=None, xcol="label"):
    """Horizontal grouped bars of paired % differences vs P0 with 95% CI whiskers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    labels = list(dict.fromkeys(tab[xcol]))
    pols = [p for p in ["P2", "P3", "P4"] if p in set(tab["a"])]
    fig, ax = plt.subplots(figsize=(8.5, min(14, max(4, 0.17 * len(labels) * len(pols) + 1.5))))
    w = 0.8 / len(pols)
    for i, p in enumerate(pols):
        g = tab[tab.a == p].set_index(xcol).reindex(labels)
        ys = np.arange(len(labels)) + (i - (len(pols) - 1) / 2) * w
        ax.barh(ys, g["diff"], height=w - 0.04, color=POLICY_COLOR[p], label=p, xerr=g["ci95"],
                error_kw=dict(ecolor=INK2, lw=0.8, capsize=0))
    ax.axvline(0, color="#52514e", lw=1)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel(title)
    ax.legend(frameon=False, fontsize=9, ncol=3, loc="lower right")
    _style(ax)
    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, dpi=160, facecolor="#fcfcfb")
    return fig


def plot_tradeoff(tab_carbon, tab_jct, outfile=None):
    """Each point = one scenario cell: x = % change in mean JCT vs P0, y = % change in dynamic carbon vs P0."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    for p in ["P2", "P3", "P4"]:
        a = tab_carbon[tab_carbon.a == p].set_index("label")
        b = tab_jct[tab_jct.a == p].set_index("label").reindex(a.index)
        ax.scatter(b["diff"], a["diff"], s=34, color=POLICY_COLOR[p], marker=POLICY_MARK[p], label=p,
                   edgecolor="#fcfcfb", linewidth=1.2, alpha=0.95)
    ax.axhline(0, color="#52514e", lw=1)
    ax.axvline(0, color="#52514e", lw=1)
    ax.set_xlabel("change in mean JCT vs P0 (%)  → slower")
    ax.set_ylabel("change in dynamic carbon vs P0 (%)  ↓ cleaner")
    ax.legend(frameon=False, fontsize=9)
    _style(ax)
    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, dpi=160, facecolor="#fcfcfb")
    return fig


def cell_label(df, cols=("arrival", "util", "comm_scale", "ci")):
    short = {"synthetic_low_var": "lowCI", "synthetic_high_var": "highCI"}
    def _ci(v):
        return short.get(v, os.path.basename(str(v)[4:]) if str(v).startswith("csv:") else v)
    return df.apply(lambda r: " / ".join(
        f"{_ci(r[c])}" if c == "ci" else (f"u{r[c]}" if c == "util" else (f"c{r[c]:g}" if c == "comm_scale" else str(r[c])))
        for c in cols), axis=1)


# ----------------------------------------------------------------------------------------------
# Reporting: tables, figures, and a compact text digest to paste back for review
# ----------------------------------------------------------------------------------------------

def sens_label(r):
    f = str(r["tag"]).split(":")[1] if ":" in str(r["tag"]) else str(r["tag"])
    if f == "slack":
        return f"slack={r['slack_lo']:g}-{r['slack_hi']:g}"
    v = r[f]
    if f == "cap_frac":
        v = "none" if v < 0 else v
    return f"{f}={v}"


def _pair_cols(df, a, b, metric, relative, label_fn):
    t = paired(df, a, b, metric, relative)
    t["label"] = t.apply(label_fn, axis=1) if len(t) else []
    t[f"{a}-{b}:{metric}"] = t["diff"].round(1).astype(str) + np.where(t["excludes_zero"], "*", "")
    return t


def make_tables(main, sens, trace=None):
    """Returns dict of DataFrames: paired tables for the main design and the sensitivity design."""
    out = {}
    comps = [("P2", "P0"), ("P3", "P0"), ("P4", "P0"), ("P4", "P2")]
    mets = [("carbon_dyn_kg", True), ("jct_mean_h", True), ("deadline_viol_frac", False), ("net_inflation", False),
            ("peak_kW", True)]
    def trace_label(r):
        return f"trace u{r['util']} c{r['comm_scale']:g} {os.path.basename(str(r['ci'])[4:]) if str(r['ci']).startswith('csv:') else str(r['ci']).replace('synthetic_', '')}"
    for name, df, lab in (("main", main, lambda r: cell_label(r.to_frame().T).iloc[0]), ("sens", sens, sens_label),
                          ("trace", trace, trace_label)):
        if df is None or df.empty:
            continue
        parts = []
        for a, b in comps + ([("P4", "P4nc"), ("P2", "P2raw")] if name == "sens" else []):
            if not (set([a, b]) <= set(df.policy)):
                continue
            for met, rel in mets:
                t = paired(df, a, b, met, rel)
                if len(t):
                    t["label"] = t.apply(lab, axis=1)
                    parts.append(t)
        out[name] = pd.concat(parts, ignore_index=True)
    return out


def save_figures(main, sens, tabs, outdir):
    os.makedirs(outdir, exist_ok=True)
    files = []
    if sens is not None and (sens.tag == "sens:comm_scale").any():
        f = os.path.join(outdir, "fig_onset_net_inflation.png")
        plot_onset(sens, "comm_scale", "net_inflation", "network runtime inflation (×, GPU-weighted)", f); files.append(f)
        f = os.path.join(outdir, "fig_onset_carbon.png")
        plot_onset(sens, "comm_scale", "carbon_dyn_kg", "dynamic carbon (kg CO2e per week)", f); files.append(f)
    if "main" in tabs:
        t = tabs["main"]
        tc = t[(t.metric == "carbon_dyn_kg") & (t.b == "P0")]
        tj = t[(t.metric == "jct_mean_h") & (t.b == "P0")]
        f = os.path.join(outdir, "fig_main_carbon_vs_p0.png")
        plot_paired_bars(tc, "dynamic carbon change vs P0 (%, paired, 95% CI)", f); files.append(f)
        f = os.path.join(outdir, "fig_main_tradeoff.png")
        plot_tradeoff(tc, tj, f); files.append(f)
    return files


def digest(main, sens, oracle_df, verify_rep, meta, tabs=None, trace=None):
    tabs = tabs or make_tables(main, sens, trace)
    L = []
    L.append("=== RESULT DIGEST (copy everything between the lines) ===")
    L.append(f"code {meta.get('code_version')} | seed {meta.get('seed')} | mode {meta.get('mode')} | "
             f"reps {meta.get('R')} | profile source: {meta.get('profile_source')} | ci scenarios: {meta.get('ci_list')}")
    L.append(f"power model: {meta.get('power_source')}")
    if meta.get("k_extension"):
        L.append(f"jobs over k>2 pods: {meta.get('k_extension')}")
    if "synthetic" in str(meta.get("profile_source")):
        L.append("!! PROFILE = SYNTHETIC PLACEHOLDER and power/fabric numbers are ILLUSTRATIVE: results below test the "
                 "pipeline and show structure; they are NOT findings until ASTRA-sim profiles + calibrated power are used.")
    nf = sum(r["status"] == "FAIL" for r in verify_rep); nw = sum(r["status"] == "WARN" for r in verify_rep)
    L.append(f"verification: {len(verify_rep)} checks, {len(verify_rep) - nf - nw} PASS, {nw} WARN, {nf} FAIL")
    for r in verify_rep:
        if r["status"] != "PASS":
            L.append(f"  [{r['status']}] {r['name']}: {r['detail']}")
    for nm, df in (("main", main), ("sens", sens)):
        if df is not None:
            L.append(f"{nm}: {len(df)} runs, censored (unfinished jobs) {(df.n_incomplete > 0).sum()}, "
                     f"mean run time {df.runtime_s.mean():.2f}s")
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 40)
    pd.set_option("display.max_rows", 500)

    def wide(t, metric, a, b, col):
        x = t[(t.metric == metric) & (t.a == a) & (t.b == b)].set_index("label")
        return x["diff"].round(1).astype(str) + np.where(x["excludes_zero"], "*", "")

    if "main" in tabs:
        t = tabs["main"]
        cols = {"carbon P2vsP0 %": wide(t, "carbon_dyn_kg", "P2", "P0", ""), "carbon P3vsP0 %": wide(t, "carbon_dyn_kg", "P3", "P0", ""),
                "carbon P4vsP0 %": wide(t, "carbon_dyn_kg", "P4", "P0", ""), "carbon P4vsP2 %": wide(t, "carbon_dyn_kg", "P4", "P2", ""),
                "JCT P2vsP0 %": wide(t, "jct_mean_h", "P2", "P0", ""), "JCT P4vsP2 %": wide(t, "jct_mean_h", "P4", "P2", ""),
                "viol P4vsP2 abs": wide(t, "deadline_viol_frac", "P4", "P2", "")}
        T = pd.DataFrame(cols)
        L.append("\nMAIN FACTORIAL: paired % differences (CRN, 95% t-interval; * = interval excludes 0). "
                 "Cell = arrival / util / comm_scale / CI scenario")
        L.append(T.to_string())
        ni = main.groupby(["comm_scale", "policy"]).net_inflation.mean().unstack().round(3)
        L.append("\nmean network runtime inflation by comm_scale (main design):")
        L.append(ni.to_string())
        if "completion_rate" in main:      # audit: unfinished jobs must stay visible next to carbon/JCT comparisons
            cr = main.groupby(["comm_scale", "policy"]).completion_rate.mean().unstack().round(3)
            L.append("\ncompletion rate (share of jobs finished inside the window) by comm_scale:"); L.append(cr.to_string())
            cg = main.groupby(["comm_scale", "policy"]).carbon_per_gpuh_g.mean().unstack().round(1)
            L.append("\ntotal carbon per COMPLETED GPU-hour (g) by comm_scale (fair when policies finish different amounts of work):"); L.append(cg.to_string())
        if "gpuh_frac_k3plus" in main:     # exposure to the k>2 extension: how much of the work the measured/extended rows govern
            kx = main.groupby(["comm_scale", "policy"]).gpuh_frac_k3plus.mean().unstack().round(3)
            L.append("\nshare of completed GPU-hours in jobs spread over 3 or more pods, by comm_scale:"); L.append(kx.to_string())
    if "sens" in tabs:
        t = tabs["sens"]
        keep = t[t.metric.isin(["carbon_dyn_kg", "jct_mean_h", "net_inflation"])]
        rows = {}
        for (a, b, met), g in keep.groupby(["a", "b", "metric"]):
            rows[f"{a}-{b} {met}"] = g.set_index("label").apply(
                lambda r: f"{r['diff']:.1f}" + ("*" if r["excludes_zero"] else "") + (f" (c{int(r['censored_reps'])})" if r["censored_reps"] else ""), axis=1)
        T = pd.DataFrame(rows)
        sel = [c for c in ["P2-P0 carbon_dyn_kg", "P3-P0 jct_mean_h", "P4-P2 carbon_dyn_kg", "P4-P2 jct_mean_h",
                           "P4-P0 carbon_dyn_kg", "P4-P4nc carbon_dyn_kg", "P2-P2raw jct_mean_h"] if c in T.columns]
        L.append("\nSENSITIVITY (one factor at a time around the base cell; (cN) = N replications censored and dropped):")
        L.append(T[sel].to_string())
        if (sens.tag == "sens:comm_scale").any():
            d = sens[sens.tag == "sens:comm_scale"].groupby(["comm_scale", "policy"]).net_inflation.mean().unstack().round(3)
            L.append("\nONSET: mean network inflation vs comm_scale (sens design):")
            L.append(d.to_string())
    if "trace" in tabs:
        t = tabs["trace"]
        T = pd.DataFrame({"carbon P2vsP0 %": wide(t, "carbon_dyn_kg", "P2", "P0", ""), "carbon P4vsP0 %": wide(t, "carbon_dyn_kg", "P4", "P0", ""),
                          "carbon P4vsP2 %": wide(t, "carbon_dyn_kg", "P4", "P2", ""), "JCT P4vsP2 %": wide(t, "jct_mean_h", "P4", "P2", "")})
        L.append("\nTRACE-DRIVEN (replication = trace window; communication intensity ASSIGNED, not measured):")
        L.append(T.to_string())
    if oracle_df is not None:
        L.append("\nBEST-KNOWN BENCHMARK on a 5-job instance (local search + heuristics; not a proof of optimality):")
        L.append(oracle_df.round(2).to_string())
    L.append("=== END DIGEST ===")
    return "\n".join(L)


# ----------------------------------------------------------------------------------------------
# Pod-size experiment: same 128 GPUs and same uplink per GPU, smaller pods, so that real trace jobs span pods
# ----------------------------------------------------------------------------------------------

def podsize_cluster(pod: int, total_gpus: int = 128) -> ClusterParams:
    """Cluster of total_gpus GPUs in pods of `pod` GPUs. Uplink and transceivers per GPU are held constant (pod=32 -> 50 GB/s, 8 ports)."""
    assert total_gpus % pod == 0
    return ClusterParams(n_pods=total_gpus // pod, gpus_per_pod=pod, ports_per_pod=max(1, 8 * pod // 32),
                         uplink_scale=pod / 32.0)


def pod_trace_summary(df):
    """Per (utilisation, communication level, CI): policy means and paired differences vs P0 and P4 vs P2 (CRN, censored reps dropped)."""
    by = ["util", "comm_scale", "ci"]
    rows = []
    for metric in ("net_inflation", "carbon_dyn_kg", "carbon_per_gpuh_g", "jct_mean_h", "peak_kW", "completion_rate"):
        for a, b in (("P2", "P0"), ("P3", "P0"), ("P4", "P0"), ("P4", "P2")):
            p = paired(df, a, b, metric=metric, by=by)
            rows.append(p[by + ["diff", "ci95", "n", "censored_reps", "excludes_zero", "a", "b", "metric"]])
    out = pd.concat(rows, ignore_index=True)
    cols = [c for c in ("net_inflation", "spread_frac", "gpuh_frac_k3plus", "completion_rate") if c in df.columns]
    means = df.groupby(by + ["policy"])[cols].mean().reset_index()
    return out, means

In [ ]:
import importlib, numpy as np, pandas as pd, scipy, matplotlib
import dcsim, dcsim_verify, dcsim_exp
for m in (dcsim, dcsim_verify, dcsim_exp):
    importlib.reload(m)
import dcsim, dcsim_verify, dcsim_exp
versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
                matplotlib=matplotlib.__version__, dcsim=dcsim.__version__)
print(versions)

## 2. Model in one page (what an examiner will ask first)

**Level of abstraction (Ch. 1.3 of the course text):** packet and wavelength detail are *not* simulated. Jobs run at *iteration* granularity; the network is a *fluid* model on pod uplinks; the optical fabric is an abstraction (capacity, port power, circuit set-up delay).

| Symbol | Meaning |
|---|---|
| `it`, `fc`, `G` | uncongested iteration time, compute fraction, per-pod-uplink GB per iteration (from the profile table) |
| `k` | pods a job spans (k=1 compact, k>=2 spread); inter-pod volume scales as 2(k-1)/k |
| `d_j = G_j / it_j` | uncongested uplink demand of job j on each pod it spans (GB/s) |
| `load_p = sum_j d_j` | demand on pod p's uplink; `rho_p = load_p / uplink_GBps` |
| `s_j = max_p curve(rho_p)` | congestion slowdown of job j (default curve = processor sharing, `max(1, rho)`; replace with an ns-3 fit) |
| `comm` | congestible (inter-pod) communication time per iteration at no congestion, hidden or exposed (profile column `comm_time_s`) |
| `it_eff = it + comm * (s-1)` | congested iteration time; conservative: assumes overlap has no slack, so even fully hidden communication becomes exposed once `s>1` |
| `P_gpu = n * (P_idle + (P_max - P_idle) * t_comp/it_eff)` | phase-dependent GPU power; more waiting = lower power but longer runtime |
| `P_total = PUE * (P_gpu_alloc + idle_GPUs*P_idle + servers + network)` | facility power; **idle GPUs and servers are always counted** |
| `Carbon = sum_t P_total(t) * dt * CI(t)` | exact piecewise integral (events include every CI hour boundary) |

**Study type:** *terminating* simulation. The cluster starts empty, 7 days of arrivals, 11 days of accounting window so every job can finish; runs where a job does not finish are flagged *censored* and dropped from paired comparisons. There is no warm-up to delete (course text Ch. 9.5), and confidence intervals come from independent replications.

**Policies.**
- **P0** baseline: FIFO + EASY backfill, first-fit GPU fill, no awareness.
- **P2** carbon-aware: shifts deferrable jobs to the lowest-forecast-CI start within deadline slack (only if the saving is at least 5%, to avoid a straw-man); network- and power-oblivious. *P2raw* = no 5% threshold.
- **P3** network-aware: starts ASAP, picks the placement minimising own runtime plus slowdown imposed on running jobs.
- **P4** joint: cost in gCO2e = carbon (with congestion-adjusted runtime and port power) + externality on running jobs + small delay price `w_delay`; compares start-now against hourly future starts using forecast free GPUs, claims of earlier queued jobs, congestion from jobs still active, and the power cap. *P4nc* = ablation without queue-claim awareness.
- **Power cap:** facility admission control for every policy (a job starts only if the cap holds).

**Randomness (Ch. 2, 10):** `SeedSequence([master, replication, stream])`, separate streams for arrivals, job attributes, carbon, and forecast noise. **All policies see the identical job stream in a replication (common random numbers)**, so differences are *paired* and reported with t-intervals.

## 3. Verification battery (run before trusting anything)

In [ ]:
t0 = time.time()
VERIFY = dcsim_verify.run_all(quick=(MODE == "quick"))
ok = dcsim_verify.print_report(VERIFY)
json.dump(VERIFY, open(f'{BASE}/results/verification_report.json', 'w'), indent=1)
print('\nverification time %.0fs' % (time.time() - t0))
assert ok, 'A verification check FAILED: do not use results until fixed (see report above).'

## 4. Job profile source (the GPU-to-network link)
Default = synthetic analytic profile (labelled). **To use ASTRA-sim results**, build `astra_profiles.csv` with one row per `(jtype, n_gpus, pclass)` where `pclass` is `compact` (one pod) or `spread` (two pods): iteration time, compute fraction (`1 - exposed_comm/total`), GB per iteration crossing each pod uplink, and (recommended) `comm_time_s` = inter-pod communication time per iteration, hidden or exposed (if absent, the exposed time is used, which understates congestion sensitivity). `dcsim.profile_row_from_astra(...)` converts ASTRA-sim's per-NPU output. **Assumption to verify:** 1 ASTRA-sim cycle = 1 ns.

In [ ]:
cluster0, power0 = dcsim.ClusterParams(k_rule=K_RULE), dcsim.PowerParams()
if os.path.exists(ASTRA_PROFILE_CSV):
    PROFILE_DF = pd.read_csv(ASTRA_PROFILE_CSV)
    print('using ASTRA-sim profile table:', ASTRA_PROFILE_CSV, len(PROFILE_DF), 'rows')
    _syn = dcsim.synthetic_profile(cluster0)
    _have = set(zip(PROFILE_DF.jtype, PROFILE_DF.n_gpus.astype(int), PROFILE_DF.pclass))
    _fill = _syn[[(a, int(b), c) not in _have for a, b, c in zip(_syn.jtype, _syn.n_gpus, _syn.pclass)]].copy()
    if len(_fill):
        _fill['source'] = 'synthetic-fill'
        PROFILE_DF = pd.concat([PROFILE_DF, _fill], ignore_index=True)
        print('WARNING: ASTRA table lacks', len(_fill), 'needed (jtype,n_gpus,pclass) keys -> filled with SYNTHETIC rows:')
        print(_fill[['jtype', 'n_gpus', 'pclass']].to_string(index=False))
else:
    PROFILE_DF = None        # run_one builds the synthetic table per scenario (comm_scale applied)
    print('NO astra_profiles.csv found -> SYNTHETIC placeholder profile (pipeline test only)')
_kx = sorted({(a, int(b), c) for a, b, c in zip(PROFILE_DF.jtype, PROFILE_DF.n_gpus, PROFILE_DF.pclass) if str(c).startswith('spread_k')}) if PROFILE_DF is not None else []
K_EXTENSION = (f"{len(_kx)} measured k>2 rows ({', '.join(f'{a} {b} {c}' for a, b, c in _kx)}); rule '{K_RULE}' beyond them" if _kx
               else f"no measured k>2 rows: every k>2 job uses rule '{K_RULE}' (run colab_cells/astra_profile_cell_kpods.py to measure them)")
print('jobs over k>2 pods:', K_EXTENSION)
prof_show = PROFILE_DF if PROFILE_DF is not None else dcsim.synthetic_profile(cluster0)
PROFILE_SOURCE = str(prof_show['source'].iloc[0]) if prof_show['source'].nunique() == 1 else 'astra-sim+synthetic-fill (' + ', '.join(f'{k}={v}' for k, v in prof_show['source'].value_counts().items()) + ')'

# Go/no-go Test 1 diagnostic: does placement change iteration time?
piv = prof_show.pivot_table(index=['jtype', 'n_gpus'], columns='pclass', values='iter_time_s')
if 'compact' in piv and 'spread' in piv:
    piv['spread/compact'] = piv['spread'] / piv['compact']
    piv['Test1 (>=1.10)'] = np.where(piv['spread/compact'] >= 1.10, 'PASS', 'no')
print(piv.round(3).to_string())
if PROFILE_SOURCE.startswith('synthetic'):
    print('\nNOTE: with a synthetic profile this ratio is true BY CONSTRUCTION. Test 1 is only evidence when the table comes from ASTRA-sim.')

# ASTRA-sim output parser self-test on the docs' published sample lines
sample = chr(10).join(f'[workload] [info] sys[{i}] finished, 64440 cycles, exposed communication 64440 cycles.' for i in range(8))
cy, ex = dcsim.parse_astra_output(sample)
print('\nparser self-test:', dcsim.profile_row_from_astra('docs_allreduce_1MB', 8, 'compact', cy, ex, 1, 0.0))

## 5. Carbon data and Test 2 (is carbon intensity variable enough to matter?)
Put hourly CSVs in `BASE/data/` named `ghana_ci.csv` or `ci_<anything>.csv` (columns `timestamp,gco2_per_kwh`). Each one becomes an extra scenario in every design, named after the file. The two synthetic scenarios are **structural stand-ins** (low vs high daily variability), not any real grid.

In [ ]:
CI_LIST = ['synthetic_low_var', 'synthetic_high_var']
rows = {k: dcsim.ci_variability(dcsim.make_ci(k, 0, 24 * 28)) for k in CI_LIST}
import glob
_real = sorted(set(([GHANA_CI_CSV] if os.path.exists(GHANA_CI_CSV) else []) + glob.glob(f'{BASE}/data/ci_*.csv')))
for _f in _real:
    _g = pd.read_csv(_f)
    _name = os.path.splitext(os.path.basename(_f))[0]
    rows[_name + ' (real, ' + str(len(_g)) + ' h)'] = dcsim.ci_variability(_g.iloc[:, 1].astype(float).to_numpy())
    CI_LIST.append('csv:' + _f)
if not _real:
    print('no real carbon-intensity CSV found (ghana_ci.csv or ci_*.csv in data/): only synthetic CI scenarios will run')
else:
    print('real carbon-intensity series used:', [os.path.basename(f) for f in _real])
tab = pd.DataFrame(rows).T
print(tab.round(2).to_string())
print('\nTest 2 reading: daily_range_over_mean well below ~0.1 means temporal shifting has little to exploit.')

## 6. GPU power: calibrate on some workloads, validate on held-out ones (optional)

In [ ]:
POWER_BASE = dcsim.PowerParams()
if os.path.exists(GPU_POWER_CSV):
    cal = dcsim_verify.calibrate_gpu_power(pd.read_csv(GPU_POWER_CSV))
    print(f"fitted P_idle={cal['P_idle_all']:.0f} W, P_max={cal['P_max_all']:.0f} W")
    print(cal['validation'].round(4).to_string())
    _pi, _pm = float(cal['P_idle_all']), float(cal['P_max_all'])
    if TARGET_GPU_MAX_W and TARGET_GPU_IDLE_W:
        print(f'TARGET GPU: idle {TARGET_GPU_IDLE_W} W and max {TARGET_GPU_MAX_W} W from datasheet/published data; the fit above only validates the LINEAR SHAPE (held-out error table)')
        _pi, _pm = float(TARGET_GPU_IDLE_W), float(TARGET_GPU_MAX_W)
    elif TARGET_GPU_MAX_W:
        _k = TARGET_GPU_MAX_W / _pm
        print(f'WARNING: only TARGET_GPU_MAX_W set: the measured idle FRACTION ({_pi/_pm:.0%}) is carried over to the target GPU. Large data-centre GPUs idle at a much lower fraction of max: set TARGET_GPU_IDLE_W too.')
        print(f'SCALING measured GPU ({_pm:.0f} W max) to target GPU max {TARGET_GPU_MAX_W} W (x{_k:.2f}): the linear SHAPE is validated on the measured GPU, the absolute watts of the target come from its datasheet')
        _pi, _pm = _pi * _k, _pm * _k
    else:
        print('TARGET_GPU_MAX_W not set: simulation uses the MEASURED GPU watts as-is (check they match the GPU you are simulating)')
    POWER_BASE.P_gpu_idle, POWER_BASE.P_gpu_max = _pi, _pm
else:
    print('no gpu_power.csv: P_gpu_idle/P_gpu_max stay ILLUSTRATIVE', POWER_BASE.P_gpu_idle, POWER_BASE.P_gpu_max,
          '(calibrate on part of your measurements, report error on the HELD-OUT part)')

## 7. Experiments (designed, replicated, checkpointed)

In [ ]:
R = 3 if MODE == "quick" else 10
RUN_TAG = f'meas5_{PROFILE_VARIANT}' if PROFILE_DF is not None else 'syn5'    # v7 (k>2 rule): new tag, so pre-v7 results are never mixed in      # new result files: old checkpoints (other comm levels / profile) are never mixed in
WORKERS = max(1, os.cpu_count() or 1)
dcsim_exp.set_comm_levels(PROFILE_DF is not None)      # measured table: comm_scale 1.0 = the measured cost
dcsim_exp.set_stress_levels(PROFILE_VARIANT == 'interleaved' and PROFILE_DF is not None)   # worst-case profile: lower load so runs can drain
# ---- input fingerprint guard (v6): checkpoint keys do not include the input files, so a results file written with a
# different profile/trace would be silently reused. Record SHA-256 fingerprints per results tag and stop on a mismatch.
import hashlib, glob
def _fp(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()[:16] if os.path.exists(path) else 'absent'
INPUTS_FP = dict(profile=_fp(ASTRA_PROFILE_CSV), trace=_fp(TRACE_CSV), ghana_ci=_fp(GHANA_CI_CSV),
                 gpu_power=_fp(GPU_POWER_CSV), dcsim=_fp('dcsim.py'))
_fp_file = f'{BASE}/results/inputs_{MODE}_{RUN_TAG}.json'
if os.path.exists(_fp_file):
    _old = json.load(open(_fp_file))
    _diff = {k: (_old.get(k), v) for k, v in INPUTS_FP.items() if _old.get(k) != v}
    if _diff:
        raise RuntimeError(f'Inputs changed since results tagged {RUN_TAG} were written: {_diff}. Move the old '
                           f'*_{MODE}_{RUN_TAG}*.csv files and {_fp_file} to another folder, then run again.')
else:
    _existing = sorted(glob.glob(f'{BASE}/results/*_{MODE}_{RUN_TAG}*.csv'))
    if _existing:
        print('WARNING: these results files already exist and predate input fingerprints; their finished runs WILL BE REUSED:')
        print(chr(10).join('  ' + os.path.basename(f) for f in _existing))
        print('  Expected only for the aligned rerun (earlier official Colab run). Otherwise stop, move them away, run again.')
    json.dump(INPUTS_FP, open(_fp_file, 'w'), indent=1)
print('results tag:', RUN_TAG, '| input fingerprints:', INPUTS_FP)
print('utilisation levels: main', dcsim_exp.UTIL_MAIN, '| sweep', dcsim_exp.UTIL_SWEEP, '| base', dcsim_exp.BASE_UTIL)
print('comm_scale levels: main', dcsim_exp.COMM_MAIN, '| sweep', dcsim_exp.COMM_SWEEP, '| base', dcsim_exp.BASE_COMM)
design_main = dcsim_exp.main_design(R=R, cis=tuple(CI_LIST))
if MODE == "quick":
    design_sens = [r for r in dcsim_exp.sensitivity_design(R=R, policies=dcsim_exp.POLICIES_MAIN)
                   if r['tag'] in ('sens:comm_scale', 'sens:util')]
else:
    design_sens = dcsim_exp.sensitivity_design(R=R, ci_extra=[c for c in CI_LIST if c.startswith('csv:')])
design_trace = dcsim_exp.trace_design(TRACE_CSV, R=R) if os.path.exists(TRACE_CSV) else []
if design_trace:
    print('trace-driven design included:', TRACE_CSV)
t0 = time.time()
kw = dict(workers=WORKERS, profile_df=PROFILE_DF, master_seed=MASTER_SEED, power_base=POWER_BASE, cluster_base=cluster0)
dcsim_exp.run_design(design_main, f'{BASE}/results/main_{MODE}_{RUN_TAG}.csv', **kw)
if design_trace:
    dcsim_exp.run_design(design_trace, f'{BASE}/results/trace_{MODE}_{RUN_TAG}.csv', **kw)
dcsim_exp.run_design(design_sens, f'{BASE}/results/sens_{MODE}_{RUN_TAG}.csv', **kw)
print('experiment time %.0fs' % (time.time() - t0))

## 7b. Pod-size experiment on the real trace (can the production workload exercise the cross-layer claim?)
Same 128 GPUs and same uplink per GPU, but pods of 32, 16 and 8 GPUs. With 32-GPU pods no trace job (at most 16 GPUs) must span pods; with 8-GPU pods every 16-GPU job must. Reported against the placement profile selected above (PROFILE_VARIANT).

In [ ]:
POD_SIZES = (32, 16, 8)
if os.path.exists(TRACE_CSV) and PROFILE_DF is not None:
    _sums, _means = [], []
    for _pod in POD_SIZES:
        _cb = dcsim_exp.podsize_cluster(_pod); _cb.k_rule = K_RULE
        _out = f'{BASE}/results/podtrace_{MODE}_{RUN_TAG}_pod{_pod}.csv'
        dcsim_exp.run_design(dcsim_exp.trace_design(TRACE_CSV, R=R), _out, workers=WORKERS, profile_df=PROFILE_DF,
                             master_seed=MASTER_SEED, power_base=POWER_BASE, cluster_base=_cb)
        _df = dcsim_exp.load_runs(_out)
        _s, _m = dcsim_exp.pod_trace_summary(_df)
        _s.insert(0, 'pod_gpus', _pod); _m.insert(0, 'pod_gpus', _pod)
        _sums.append(_s); _means.append(_m)
    POD_SUM = pd.concat(_sums, ignore_index=True); POD_MEANS = pd.concat(_means, ignore_index=True)
    POD_SUM.to_csv(f'{BASE}/results/podtrace_paired_{MODE}_{RUN_TAG}.csv', index=False)
    POD_MEANS.to_csv(f'{BASE}/results/podtrace_means_{MODE}_{RUN_TAG}.csv', index=False)
    print('POD-SIZE EXPERIMENT, profile =', PROFILE_VARIANT, '| trace =', TRACE_CSV)
    print('\nMean network inflation / fraction of jobs spanning pods / completion rate, by policy:')
    print(POD_MEANS.round(3).to_string(index=False))
    print('\nPaired differences (percent; 95% t-interval half-width; reps censored):')
    _t = POD_SUM[POD_SUM.metric.isin(['net_inflation', 'carbon_per_gpuh_g', 'jct_mean_h', 'peak_kW'])]
    print(_t.round(2).to_string(index=False))
else:
    print('skipped: needs trace.csv and a measured profile table')

In [ ]:
# Best-known benchmark for the P4 heuristic on a 5-job instance (local search; not a proof of optimality)
t0 = time.time()
ORACLE_DF, _ = dcsim_exp.oracle_benchmark(comm_scale=12.0, restarts=(40 if MODE == 'quick' else 150))
print(ORACLE_DF.round(2).to_string())
print('benchmark time %.0fs' % (time.time() - t0))

## 8. Analysis: paired differences with 95% intervals, mechanism plots

In [ ]:
main = dcsim_exp.load_runs(f'{BASE}/results/main_{MODE}_{RUN_TAG}.csv')
sens = dcsim_exp.load_runs(f'{BASE}/results/sens_{MODE}_{RUN_TAG}.csv')
TRACE = dcsim_exp.load_runs(f'{BASE}/results/trace_{MODE}_{RUN_TAG}.csv') if os.path.exists(f'{BASE}/results/trace_{MODE}_{RUN_TAG}.csv') else None
TABS = dcsim_exp.make_tables(main, sens, TRACE)
for k, v in TABS.items():
    v.to_csv(f'{BASE}/results/paired_{k}_{MODE}_{RUN_TAG}.csv', index=False)
FIGS = dcsim_exp.save_figures(main, sens, TABS, f'{BASE}/figures/{MODE}_{RUN_TAG}')
def show_img(p):
    try:
        from IPython.display import Image, display
        display(Image(filename=p))
    except Exception:
        print('figure saved:', p)
for p in FIGS:
    show_img(p)

## 9. Bundle and digest (copy the digest block back to Claude)

In [ ]:
import zipfile, subprocess
_ps = ('published H100 node data (arXiv 2506.14551), linear shape validated on a T4' if (os.path.exists(GPU_POWER_CSV) and TARGET_GPU_MAX_W and TARGET_GPU_IDLE_W) else ('MEASURED on the calibration GPU as-is' if os.path.exists(GPU_POWER_CSV) else 'ILLUSTRATIVE placeholders'))
META = dict(power_source=f'GPU idle {POWER_BASE.P_gpu_idle:.0f} W / max {POWER_BASE.P_gpu_max:.0f} W ({_ps}); server {POWER_BASE.P_server:.0f} W ASSUMED; PUE {POWER_BASE.PUE} ASSUMED', code_version=dcsim.__version__, seed=MASTER_SEED, mode=MODE, R=R, profile_source=PROFILE_SOURCE,
            k_extension=K_EXTENSION, ci_list=', '.join(os.path.basename(c) if c.startswith('csv:') else c for c in CI_LIST))
DIGEST = dcsim_exp.digest(main, sens, ORACLE_DF, VERIFY, META, TABS, TRACE)
DIGEST = DIGEST.replace(chr(10), chr(10) + f'profile variant: {PROFILE_VARIANT} | results tag: {RUN_TAG} | input fingerprints: {INPUTS_FP}' + chr(10), 1)
open(f'{BASE}/results/DIGEST_{MODE}_{RUN_TAG}.txt', 'w').write(DIGEST)

params = dict(cluster=dcsim.asdict(cluster0), power=dcsim.asdict(POWER_BASE), fabrics=dcsim.FABRICS,
              job_types=dcsim.JOB_TYPES, meta=META, versions=versions,
              profile_table=prof_show.to_dict(orient='records'))
json.dump(params, open(f'{BASE}/results/params_{MODE}_{RUN_TAG}.json', 'w'), indent=1, default=str)
try:
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
except Exception:
    freeze = ''
open(f'{BASE}/results/pip_freeze.txt', 'w').write(freeze)

REPRO = f'''# Reproducibility statement (course text Ch. 13.4 checklist), mode={MODE}
1. Model, assumptions, abstractions: terminating DES; fluid pod-uplink congestion; optical fabric abstracted to capacity, port power and set-up delay; constant PUE; no preemption or migration; single site.
2. Simulator and version: dcsim {dcsim.__version__}; every parameter including defaults is in params_{MODE}_{RUN_TAG}.json; Python/package versions in pip_freeze.txt.
3. Seeds, replications, warm-up, run length: master seed {MASTER_SEED}; SeedSequence([seed, replication, stream]); R={R} replications; no warm-up (terminating); arrivals over 168 h, accounting window 264 h.
4. Confidence intervals: t-interval over independent replications; paired differences under common random numbers; runs with unfinished jobs are censored and excluded from paired comparisons (counts reported).
5. Validation experiments shown: M/M/c against Erlang-C; Little's law; hand-computed energy and carbon cases (constant and hourly-varying CI); congestion degenerate cases; energy conservation. NOT validated against a real facility; profile source = {PROFILE_SOURCE}.
6. Limitations: see digest header and the notebook limitations section. Workload mix, durations, slack, power and fabric numbers are ILLUSTRATIVE unless replaced by the optional data files.
'''
open(f'{BASE}/results/REPRODUCIBILITY_STATEMENT_{RUN_TAG}.md', 'w').write(REPRO)

zpath = f'{BASE}/bundles/results_{MODE}_{RUN_TAG}_{time.strftime("%Y%m%d_%H%M%S")}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('results', 'figures'):
        for root, _dirs, fns in os.walk(f'{BASE}/{folder}'):
            for fn in fns:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, BASE))
    for fn in ('dcsim.py', 'dcsim_verify.py', 'dcsim_exp.py'):
        z.write(fn, f'code/{fn}')
print('bundle:', zpath)
try:
    from google.colab import files
    files.download(zpath)
except Exception:
    pass
print()
print(DIGEST)

## 10. Where each earlier gap is closed

| Gap raised | Closed by |
|---|---|
| "Optical" undefined | Fabric abstraction (`FABRICS`: capacity, port power, set-up delay), no wavelength-level claims |
| No hypothesis or mechanism | Congestion model + metrics `net_inflation`, `spread_frac`, `ci_at_start_gpuw`; onset sweep over `comm_scale` |
| No power-aware element | Facility cap (admission control, tested), peak power, idle GPU and server power always counted |
| P4 undefined | `P4` cost function and algorithm in the source; `P4nc` ablation; best-known benchmark |
| Equation 8 / routing variable | Per-pod uplink load `load_p` and slowdown `s_j` (model table above) |
| Ghana variability unknown | Test 2 table + `ghana_ci.csv` hook; synthetic low/high variability contrast |
| Traces lack communication data | `trace.csv` hook; communication intensity assigned and stated; `comm_scale` sensitivity |
| Validation vs calibration | `calibrate_gpu_power` fits on some workloads and reports error on held-out ones |
| Single seed / no intervals (course text Ch. 10-11) | Replications, SeedSequence streams, t-intervals, CRN pairing, censoring flag |
| Terminating vs steady state (Ch. 9.5) | Declared terminating; no warm-up; censoring handled |
| Verification battery (Ch. 13.2) | 16 checks (14 in quick mode) incl. M/M/c vs Erlang-C, Little's law, hand cases, conservation, degenerate cases, generators |
| Traffic model drives conclusions (Ch. 10.5) | Poisson, bursty (MMPP), diurnal arrivals and trace-driven mode; arrival factor in the designs |
| Factorial design, not wandering (Ch. 12.4) | `main_design` (full factorial, few levels) plus one-factor sweeps; all raw results saved with configuration |
| Idle power missing | Idle GPUs and servers always counted; `carbon_dyn_kg` isolates the policy-dependent part |
| Reporting checklist (Ch. 13.4) | Auto-generated `REPRODUCIBILITY_STATEMENT.md`, `params.json`, `pip_freeze.txt`, versioned code in the zip |
| Discretisation insensitivity | Scheduler tick halving check (full mode) |

## 11. Limitations (written before a reviewer writes them)
- **Profile table.** Synthetic until ASTRA-sim rows are supplied. ASTRA-sim reports cycles; the 1 cycle = 1 ns conversion is an assumption to verify.
- **Congestion** is a fluid, bottleneck-pod approximation using uncongested demand; intra-pod fabric is non-blocking; jobs over k>2 pods use measured rows where present, otherwise a flat-ring extension of the 2-pod row (`K_RULE`); uneven splits share one row per (job type, GPUs, k). Calibrate `congestion_pts` from ns-3 runs of concurrent jobs.
- **Optical layer** is an abstraction; there is no wavelength assignment, no physical-layer model, and reconfiguration delay is paid once per spread job.
- **Workloads.** Job mix, durations, deferrable fraction and slack are ILLUSTRATIVE; traces lack communication data, so intensity is assigned.
- **Forecasts** are perfect by default (`forecast_sigma` adds multiplicative noise in the sensitivity design).
- **Scope.** Single site, no preemption, migration or checkpointing, constant PUE, training jobs only.
- **Benchmark** is a best-known solution from local search on 5 jobs, not a proof of optimality.